In [ ]:
# Paths. Change them here, or set the environment variables DATA_DIR and OUT_DIR.
import os
from pathlib import Path
DATA_DIR = os.environ.get("DATA_DIR", "../data")      # put the Kaggle CSV here
OUT_DIR = os.environ.get("OUT_DIR", "../outputs")     # tables and figures are written here
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("Data folder  :", Path(DATA_DIR).resolve())
print("Output folder:", Path(OUT_DIR).resolve())

# Leakage-Aware Feature Group Analysis for Dry Eye Disease Prediction

## Research Objective

This study investigates machine-learning-based Dry Eye Disease (DED) classification with particular attention to the contribution of different groups of predictors.

The workflow includes:

1. Data quality auditing
2. Blood pressure parsing and data encoding
3. Exploratory data analysis
4. Feature grouping
5. Baseline machine-learning models
6. Feature-group ablation analysis
7. SHAP-based explainability
8. Feature-importance stability analysis
9. Calibration analysis
10. Statistical comparison and interpretation

A central objective is to determine whether predictive performance is driven primarily by direct ocular symptoms or whether demographic, sleep, lifestyle, and digital-exposure factors provide independent predictive information.

## 1. Environment Setup

The required Python libraries are imported and a fixed random seed is defined to make the experiments reproducible.

GPU acceleration is not required because the dataset is tabular and contains approximately 20,000 observations.

In [ ]:
# ============================================================
# STEP 1: IMPORT LIBRARIES AND CHECK ENVIRONMENT
# ============================================================

try:
    import os
    import sys
    import warnings
    from pathlib import Path

    import numpy as np
    import pandas as pd

    import matplotlib.pyplot as plt
    import seaborn as sns

    import sklearn

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------
    RANDOM_STATE = 42
    np.random.seed(RANDOM_STATE)

    # Reduce unnecessary warning clutter
    warnings.filterwarnings("ignore")

    print("=" * 70)
    print("ENVIRONMENT CHECK")
    print("=" * 70)

    print(f"Python version       : {sys.version.split()[0]}")
    print(f"Pandas version       : {pd.__version__}")
    print(f"NumPy version        : {np.__version__}")
    print(f"Scikit-learn version : {sklearn.__version__}")

    # Data folder check
    if Path(DATA_DIR).exists():
        print("\n✅ Data folder detected.")
    else:
        print("\n⚠️ Data folder not found. Create it and add the Kaggle CSV.")

    print("\n✅ STEP 1 COMPLETED SUCCESSFULLY.")

except Exception as e:
    print("\n❌ STEP 1 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 2. Dataset Loading

The notebook automatically searches Kaggle's input directory for the Dry Eye Disease dataset.

Instead of relying only on the filename, the dataset is validated using several expected column names to reduce the risk of accidentally loading the wrong CSV file.

In [ ]:
# ============================================================
# STEP 2: FIND AND LOAD THE DRY EYE DATASET
# ============================================================

try:
    # --------------------------------------------------------
    # Expected columns used to identify the correct dataset
    # --------------------------------------------------------
    required_columns = {
        "Gender",
        "Age",
        "Sleep duration",
        "Blood pressure",
        "Average screen time",
        "Discomfort Eye-strain",
        "Redness in eye",
        "Itchiness/Irritation in eye",
        "Dry Eye Disease"
    }

    # --------------------------------------------------------
    # Search all CSV files in the data folder
    # --------------------------------------------------------
    csv_files = list(Path(DATA_DIR).rglob("*.csv"))

    print(f"CSV files found: {len(csv_files)}")
    print("-" * 70)

    if len(csv_files) == 0:
        raise FileNotFoundError(
            "No CSV files were found. "
            "Please download the Dry Eye Disease CSV from Kaggle and put it in the data folder."
        )

    valid_candidates = []

    for file_path in csv_files:

        try:
            temp_df = pd.read_csv(file_path)

            if required_columns.issubset(set(temp_df.columns)):
                valid_candidates.append(file_path)

        except Exception:
            # Ignore unrelated or unreadable CSV files
            pass

    # --------------------------------------------------------
    # Validate result
    # --------------------------------------------------------
    if len(valid_candidates) == 0:
        raise FileNotFoundError(
            "No CSV matching the expected Dry Eye dataset structure was found."
        )

    if len(valid_candidates) > 1:
        print("⚠️ Multiple matching datasets found:")
        for path in valid_candidates:
            print(" -", path)

        print("\nThe first matching dataset will be used.")

    DATA_PATH = valid_candidates[0]

    print(f"\nSelected dataset:\n{DATA_PATH}")

    # --------------------------------------------------------
    # Load dataset
    # --------------------------------------------------------
    df_raw = pd.read_csv(DATA_PATH)

    if df_raw.empty:
        raise ValueError("The selected CSV contains no observations.")

    print("\n✅ Dataset loaded successfully.")
    print(f"Rows    : {df_raw.shape[0]:,}")
    print(f"Columns : {df_raw.shape[1]}")

    print("\nFirst 5 observations:")
    display(df_raw.head())

    print("\n✅ STEP 2 COMPLETED SUCCESSFULLY.")

except Exception as e:
    print("\n❌ STEP 2 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 3. Data Quality Audit

Before preprocessing or machine learning, the dataset is examined for:

- number of observations and attributes
- duplicate records
- missing values
- constant features
- numerical and categorical variables
- unexpected categorical values
- target class distribution
- numerical feature ranges
- validity of the blood-pressure format

No observations are removed during this stage.

In [ ]:
# ============================================================
# STEP 3: DATA QUALITY AUDIT
# CHECKPOINT 1A
# ============================================================

try:
    df = df_raw.copy()

    TARGET_COLUMN = "Dry Eye Disease"

    print("=" * 80)
    print("DRY EYE DATASET QUALITY AUDIT")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Shape
    # --------------------------------------------------------
    print(f"\nRows    : {df.shape[0]:,}")
    print(f"Columns : {df.shape[1]}")

    # --------------------------------------------------------
    # 2. Duplicate rows
    # --------------------------------------------------------
    duplicate_count = df.duplicated().sum()

    print(f"\nDuplicate rows: {duplicate_count:,}")

    # --------------------------------------------------------
    # 3. Missing values
    # --------------------------------------------------------
    total_missing = df.isna().sum().sum()

    print(f"Total missing values: {total_missing:,}")

    missing_by_column = (
        df.isna()
        .sum()
        .sort_values(ascending=False)
    )

    missing_columns = missing_by_column[
        missing_by_column > 0
    ]

    if len(missing_columns) == 0:
        print("✅ No standard missing values detected.")
    else:
        print("\nColumns containing missing values:")
        display(
            missing_columns.to_frame(
                name="Missing_Count"
            )
        )

    # --------------------------------------------------------
    # 4. Constant columns
    # --------------------------------------------------------
    constant_columns = [
        col
        for col in df.columns
        if df[col].nunique(dropna=False) <= 1
    ]

    print(f"\nConstant columns: {len(constant_columns)}")

    if constant_columns:
        print(constant_columns)
    else:
        print("✅ No constant columns detected.")

    # --------------------------------------------------------
    # 5. Data types
    # --------------------------------------------------------
    numeric_columns_raw = (
        df.select_dtypes(include="number")
        .columns
        .tolist()
    )

    object_columns_raw = (
        df.select_dtypes(include=["object", "category"])
        .columns
        .tolist()
    )

    print("\nFeature type summary:")
    print(f"Numerical columns   : {len(numeric_columns_raw)}")
    print(f"Object columns      : {len(object_columns_raw)}")

    # --------------------------------------------------------
    # 6. Target validation
    # --------------------------------------------------------
    if TARGET_COLUMN not in df.columns:
        raise KeyError(
            f"Target column '{TARGET_COLUMN}' was not found."
        )

    target_values = set(
        df[TARGET_COLUMN]
        .astype(str)
        .str.strip()
        .unique()
    )

    print("\nTarget values:")
    print(target_values)

    if target_values != {"Y", "N"}:
        raise ValueError(
            f"Unexpected target values detected: {target_values}"
        )

    # --------------------------------------------------------
    # 7. Target distribution
    # --------------------------------------------------------
    target_counts = (
        df[TARGET_COLUMN]
        .value_counts()
    )

    target_percent = (
        df[TARGET_COLUMN]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    target_summary = pd.DataFrame({
        "Count": target_counts,
        "Percentage": target_percent
    })

    print("\nTarget distribution:")
    display(target_summary)

    # --------------------------------------------------------
    # 8. Show categorical values
    #
    # Blood pressure is excluded here because it contains
    # many distinct valid combinations such as 120/80.
    # --------------------------------------------------------
    print("\nCategorical value check:")
    print("-" * 80)

    for col in object_columns_raw:

        if col == "Blood pressure":
            continue

        unique_values = sorted(
            df[col]
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )

        print(f"{col}: {unique_values}")

    # --------------------------------------------------------
    # 9. Numerical ranges
    # --------------------------------------------------------
    numeric_range_table = pd.DataFrame({
        "Minimum": df[numeric_columns_raw].min(),
        "Maximum": df[numeric_columns_raw].max(),
        "Mean": df[numeric_columns_raw].mean()
    }).round(3)

    print("\nNumerical feature ranges:")
    display(numeric_range_table)

    # --------------------------------------------------------
    # 10. Complete column list
    # --------------------------------------------------------
    print("\nComplete column list:")
    print("-" * 80)

    for i, column in enumerate(df.columns, start=1):
        print(f"{i:02d}. {column}")

    print("\n" + "=" * 80)
    print("✅ CHECKPOINT 1A COMPLETED SUCCESSFULLY.")
    print("=" * 80)

except Exception as e:
    print("\n❌ CHECKPOINT 1A FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 4. Blood Pressure Validation

Blood pressure is stored as a text variable in the form `systolic/diastolic`.

Before converting it into numerical predictors, every record is checked for correct formatting. Systolic and diastolic values are then extracted separately.

A pulse-pressure check is also performed:

**Pulse Pressure = Systolic Pressure − Diastolic Pressure**

Records with non-positive pulse pressure are flagged for inspection rather than automatically deleted.

In [ ]:
# ============================================================
# STEP 4: BLOOD PRESSURE FORMAT AND QUALITY CHECK
# CHECKPOINT 1B
# ============================================================

try:
    BP_COLUMN = "Blood pressure"

    # --------------------------------------------------------
    # 1. Convert to clean strings
    # --------------------------------------------------------
    bp_text = (
        df[BP_COLUMN]
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------------
    # 2. Validate format
    #
    # Expected examples:
    # 120/80
    # 105/70
    # --------------------------------------------------------
    valid_bp_pattern = bp_text.str.match(
        r"^\d{2,3}/\d{2,3}$"
    )

    invalid_bp_count = (~valid_bp_pattern).sum()

    print("=" * 80)
    print("BLOOD PRESSURE VALIDATION")
    print("=" * 80)

    print(
        f"\nCorrectly formatted records : "
        f"{valid_bp_pattern.sum():,}"
    )

    print(
        f"Invalidly formatted records : "
        f"{invalid_bp_count:,}"
    )

    if invalid_bp_count > 0:

        print("\nInvalid blood-pressure values:")

        display(
            df.loc[
                ~valid_bp_pattern,
                [BP_COLUMN]
            ].value_counts()
        )

        raise ValueError(
            "Invalid blood-pressure formats were found. "
            "Do not continue until inspected."
        )

    # --------------------------------------------------------
    # 3. Extract systolic and diastolic components
    # --------------------------------------------------------
    bp_parts = bp_text.str.extract(
        r"^(?P<Systolic_BP>\d{2,3})/"
        r"(?P<Diastolic_BP>\d{2,3})$"
    )

    bp_parts = bp_parts.astype(int)

    # --------------------------------------------------------
    # 4. Pulse pressure for QUALITY CHECK only
    #
    # We are NOT adding pulse pressure as a modeling feature
    # at this stage.
    # --------------------------------------------------------
    pulse_pressure = (
        bp_parts["Systolic_BP"]
        - bp_parts["Diastolic_BP"]
    )

    # --------------------------------------------------------
    # 5. Summary
    # --------------------------------------------------------
    bp_summary = pd.DataFrame({
        "Minimum": [
            bp_parts["Systolic_BP"].min(),
            bp_parts["Diastolic_BP"].min(),
            pulse_pressure.min()
        ],
        "Maximum": [
            bp_parts["Systolic_BP"].max(),
            bp_parts["Diastolic_BP"].max(),
            pulse_pressure.max()
        ],
        "Mean": [
            bp_parts["Systolic_BP"].mean(),
            bp_parts["Diastolic_BP"].mean(),
            pulse_pressure.mean()
        ]
    }, index=[
        "Systolic BP",
        "Diastolic BP",
        "Pulse Pressure"
    ]).round(2)

    print("\nBlood-pressure summary:")
    display(bp_summary)

    # --------------------------------------------------------
    # 6. Flag unusual records where
    # systolic <= diastolic
    # --------------------------------------------------------
    non_positive_pulse_mask = (
        pulse_pressure <= 0
    )

    non_positive_count = (
        non_positive_pulse_mask.sum()
    )

    print(
        f"\nRecords with systolic <= diastolic: "
        f"{non_positive_count}"
    )

    if non_positive_count > 0:

        print(
            "\n⚠️ These records are flagged for inspection."
        )

        flagged_bp_records = (
            df.loc[
                non_positive_pulse_mask,
                [BP_COLUMN, TARGET_COLUMN]
            ]
            .copy()
        )

        flagged_bp_records["Systolic_BP"] = (
            bp_parts.loc[
                non_positive_pulse_mask,
                "Systolic_BP"
            ]
        )

        flagged_bp_records["Diastolic_BP"] = (
            bp_parts.loc[
                non_positive_pulse_mask,
                "Diastolic_BP"
            ]
        )

        flagged_bp_records["Pulse_Pressure"] = (
            pulse_pressure[
                non_positive_pulse_mask
            ]
        )

        display(flagged_bp_records)

        print("\nFrequency of flagged BP values:")

        display(
            flagged_bp_records[
                BP_COLUMN
            ].value_counts()
        )

    else:
        print(
            "✅ No non-positive pulse-pressure records detected."
        )

    print("\n✅ CHECKPOINT 1B COMPLETED SUCCESSFULLY.")

except Exception as e:
    print("\n❌ CHECKPOINT 1B FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 5. Blood Pressure Parsing and Binary Encoding

The cleaned analysis dataset is created as follows:

- `Blood pressure` is separated into systolic and diastolic numerical attributes.
- The original combined blood-pressure text column is removed.
- Binary Y/N predictors are encoded as `1/0`.
- Gender is represented using a binary `Gender_Male` variable.
- Dry Eye Disease is encoded as:
  - `0 = No`
  - `1 = Yes`

The original unmodified dataframe remains available as `df_raw`.

In [ ]:
# ============================================================
# STEP 5: PARSE BLOOD PRESSURE AND ENCODE BINARY VARIABLES
# CHECKPOINT 1C
# ============================================================

try:
    # --------------------------------------------------------
    # Start from the original dataset again
    # --------------------------------------------------------
    df_clean = df_raw.copy()

    # --------------------------------------------------------
    # 1. Split Blood Pressure
    # --------------------------------------------------------
    bp_split = (
        df_clean["Blood pressure"]
        .astype(str)
        .str.strip()
        .str.extract(
            r"^(?P<Systolic_BP>\d{2,3})/"
            r"(?P<Diastolic_BP>\d{2,3})$"
        )
    )

    if bp_split.isna().any().any():
        raise ValueError(
            "At least one blood-pressure value could not be parsed."
        )

    df_clean["Systolic BP"] = (
        bp_split["Systolic_BP"]
        .astype(int)
    )

    df_clean["Diastolic BP"] = (
        bp_split["Diastolic_BP"]
        .astype(int)
    )

    # Remove original combined BP column
    df_clean.drop(
        columns=["Blood pressure"],
        inplace=True
    )

    # --------------------------------------------------------
    # 2. Identify Y/N predictors
    #
    # Do NOT include the target here yet.
    # --------------------------------------------------------
    yn_predictors = [
        "Sleep disorder",
        "Wake up during night",
        "Feel sleepy during day",
        "Caffeine consumption",
        "Alcohol consumption",
        "Smoking",
        "Medical issue",
        "Ongoing medication",
        "Smart device before bed",
        "Blue-light filter",
        "Discomfort Eye-strain",
        "Redness in eye",
        "Itchiness/Irritation in eye"
    ]

    # --------------------------------------------------------
    # 3. Validate Y/N predictor values before mapping
    # --------------------------------------------------------
    for col in yn_predictors:

        values = set(
            df_clean[col]
            .astype(str)
            .str.strip()
            .unique()
        )

        if not values.issubset({"Y", "N"}):
            raise ValueError(
                f"Unexpected values in '{col}': {values}"
            )

        # Clean spaces and convert
        df_clean[col] = (
            df_clean[col]
            .astype(str)
            .str.strip()
            .map({
                "N": 0,
                "Y": 1
            })
        )

    # --------------------------------------------------------
    # 4. Gender encoding
    #
    # Instead of calling it simply 0/1,
    # create an explicitly named feature.
    # --------------------------------------------------------
    gender_values = set(
        df_clean["Gender"]
        .astype(str)
        .str.strip()
        .unique()
    )

    if gender_values != {"M", "F"}:
        raise ValueError(
            f"Unexpected Gender values: {gender_values}"
        )

    df_clean["Gender_Male"] = (
        df_clean["Gender"]
        .astype(str)
        .str.strip()
        .map({
            "F": 0,
            "M": 1
        })
    )

    df_clean.drop(
        columns=["Gender"],
        inplace=True
    )

    # --------------------------------------------------------
    # 5. Encode target
    # --------------------------------------------------------
    target_values = set(
        df_clean["Dry Eye Disease"]
        .astype(str)
        .str.strip()
        .unique()
    )

    if target_values != {"Y", "N"}:
        raise ValueError(
            f"Unexpected target values: {target_values}"
        )

    df_clean["Dry_Eye"] = (
        df_clean["Dry Eye Disease"]
        .astype(str)
        .str.strip()
        .map({
            "N": 0,
            "Y": 1
        })
    )

    # Remove original target string
    df_clean.drop(
        columns=["Dry Eye Disease"],
        inplace=True
    )

    # --------------------------------------------------------
    # 6. Integrity checks
    # --------------------------------------------------------
    remaining_object_columns = (
        df_clean
        .select_dtypes(include=["object", "category"])
        .columns
        .tolist()
    )

    missing_after_cleaning = (
        df_clean
        .isna()
        .sum()
        .sum()
    )

    print("=" * 80)
    print("CLEAN DATASET SUMMARY")
    print("=" * 80)

    print(
        f"\nOriginal shape : {df_raw.shape}"
    )

    print(
        f"Cleaned shape  : {df_clean.shape}"
    )

    print(
        f"Missing values : {missing_after_cleaning}"
    )

    print(
        f"Remaining object columns: "
        f"{remaining_object_columns}"
    )

    # --------------------------------------------------------
    # 7. Target distribution after encoding
    # --------------------------------------------------------
    encoded_target_summary = pd.DataFrame({
        "Count": df_clean["Dry_Eye"]
                    .value_counts()
                    .sort_index(),

        "Percentage":
            df_clean["Dry_Eye"]
            .value_counts(normalize=True)
            .sort_index()
            .mul(100)
            .round(2)
    })

    encoded_target_summary.index = [
        "0 = No Dry Eye",
        "1 = Dry Eye"
    ]

    print("\nEncoded target distribution:")
    display(encoded_target_summary)

    # --------------------------------------------------------
    # 8. Show final columns and dtypes
    # --------------------------------------------------------
    print("\nCleaned dataset columns:")

    for i, col in enumerate(
        df_clean.columns,
        start=1
    ):
        print(
            f"{i:02d}. {col} "
            f"[{df_clean[col].dtype}]"
        )

    # --------------------------------------------------------
    # 9. Final assertions
    # --------------------------------------------------------
    if len(remaining_object_columns) != 0:
        raise TypeError(
            "Some categorical columns remain unencoded."
        )

    if missing_after_cleaning != 0:
        raise ValueError(
            "Missing values appeared during preprocessing."
        )

    if set(df_clean["Dry_Eye"].unique()) != {0, 1}:
        raise ValueError(
            "Target encoding failed."
        )

    print("\n✅ All variables are now numeric.")
    print("✅ Binary encoding completed successfully.")
    print("✅ Original df_raw remains unchanged.")

    print("\n" + "=" * 80)
    print("✅ CHECKPOINT 1C COMPLETED SUCCESSFULLY.")
    print("=" * 80)

except Exception as e:
    print("\n❌ CHECKPOINT 1C FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### Saving the Cleaned Dataset

The cleaned numerical dataset is saved to the Kaggle working directory for reproducibility and later analysis.

In [ ]:
# ============================================================
# STEP 6: SAVE CLEANED DATASET
# ============================================================

try:
    CLEAN_DATA_PATH = (
        f"{OUT_DIR}/"
        "dry_eye_cleaned.csv"
    )

    df_clean.to_csv(
        CLEAN_DATA_PATH,
        index=False
    )

    print("✅ Cleaned dataset saved successfully.")
    print(f"Location: {CLEAN_DATA_PATH}")

except Exception as e:
    print("\n❌ SAVE FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## Final Preprocessing Verification

Before exploratory analysis, the cleaned dataset is verified for:

- expected dimensions
- missing values
- remaining non-numeric columns
- target balance
- parsed blood-pressure ranges
- unusual blood-pressure observations

This ensures that the dataset is ready for statistical analysis and machine-learning experiments.

In [ ]:
# ============================================================
# CHECKPOINT 1 FINAL:
# VERIFY CLEANED DATASET BEFORE EDA
# ============================================================

try:
    print("=" * 75)
    print("FINAL PREPROCESSING VERIFICATION")
    print("=" * 75)

    # --------------------------------------------------------
    # Basic cleaned dataset checks
    # --------------------------------------------------------
    print(f"\nCleaned shape        : {df_clean.shape}")
    print(
        f"Missing values      : "
        f"{df_clean.isna().sum().sum()}"
    )

    object_cols = (
        df_clean
        .select_dtypes(include=["object", "category"])
        .columns
        .tolist()
    )

    print(
        f"Remaining object cols: {object_cols}"
    )

    # --------------------------------------------------------
    # Target check
    # --------------------------------------------------------
    print("\nTarget distribution:")

    target_check = pd.DataFrame({
        "Count":
            df_clean["Dry_Eye"]
            .value_counts()
            .sort_index(),

        "Percentage":
            df_clean["Dry_Eye"]
            .value_counts(normalize=True)
            .sort_index()
            .mul(100)
            .round(2)
    })

    target_check.index = [
        "0 = No Dry Eye",
        "1 = Dry Eye"
    ]

    display(target_check)

    # --------------------------------------------------------
    # Blood pressure checks
    # --------------------------------------------------------
    print("\nBlood pressure ranges:")

    print(
        f"Systolic BP  : "
        f"{df_clean['Systolic BP'].min()} "
        f"to {df_clean['Systolic BP'].max()}"
    )

    print(
        f"Diastolic BP : "
        f"{df_clean['Diastolic BP'].min()} "
        f"to {df_clean['Diastolic BP'].max()}"
    )

    # --------------------------------------------------------
    # Check unusual BP records
    # --------------------------------------------------------
    unusual_bp = df_clean[
        df_clean["Systolic BP"]
        <=
        df_clean["Diastolic BP"]
    ]

    print(
        f"\nRecords with Systolic <= Diastolic: "
        f"{len(unusual_bp)}"
    )

    if len(unusual_bp) > 0:

        print("\nFrequency of unusual BP combinations:")

        display(
            unusual_bp[
                ["Systolic BP", "Diastolic BP"]
            ]
            .value_counts()
            .to_frame(name="Count")
        )

    # --------------------------------------------------------
    # Final readiness checks
    # --------------------------------------------------------
    if df_clean.isna().sum().sum() != 0:
        raise ValueError(
            "Missing values exist after cleaning."
        )

    if len(object_cols) != 0:
        raise TypeError(
            "Non-numeric columns remain after encoding."
        )

    if set(df_clean["Dry_Eye"].unique()) != {0, 1}:
        raise ValueError(
            "Target is not correctly encoded as 0/1."
        )

    print("\n✅ Dataset is ready for EDA.")
    print("✅ CHECKPOINT 1 FINAL PASSED.")

except Exception as e:
    print("\n❌ CHECKPOINT 1 FINAL FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

FINAL PREPROCESSING VERIFICATION
===========================================================================

Cleaned shape        : (20000, 27)
Missing values      : 0
Remaining object cols: []

Target distribution:
Count	Percentage
0 = No Dry Eye	6963	34.82
1 = Dry Eye	13037	65.18

Blood pressure ranges:
Systolic BP  : 90 to 140
Diastolic BP : 60 to 90

Records with Systolic <= Diastolic: 11

Frequency of unusual BP combinations:
Count
Systolic BP	Diastolic BP	
90	90	11

✅ Dataset is ready for EDA.
✅ CHECKPOINT 1 FINAL PASSED.

In [ ]:
# ============================================================
# STEP 7: REMOVE INVALID BLOOD-PRESSURE RECORDS
# ============================================================

try:
    # --------------------------------------------------------
    # Keep a copy of the fully encoded dataset
    # before any row filtering.
    # --------------------------------------------------------
    df_before_bp_filter = df_clean.copy()

    # --------------------------------------------------------
    # A valid record should have:
    # Systolic BP > Diastolic BP
    # --------------------------------------------------------
    invalid_bp_mask = (
        df_clean["Systolic BP"]
        <=
        df_clean["Diastolic BP"]
    )

    removed_records = df_clean.loc[
        invalid_bp_mask
    ].copy()

    # --------------------------------------------------------
    # Remove invalid BP observations
    # --------------------------------------------------------
    df_model = (
        df_clean.loc[~invalid_bp_mask]
        .copy()
        .reset_index(drop=True)
    )

    print("=" * 75)
    print("FINAL BLOOD-PRESSURE FILTER")
    print("=" * 75)

    print(
        f"\nOriginal observations : "
        f"{len(df_clean):,}"
    )

    print(
        f"Removed observations  : "
        f"{len(removed_records):,}"
    )

    print(
        f"Remaining observations: "
        f"{len(df_model):,}"
    )

    removal_percentage = (
        len(removed_records)
        / len(df_clean)
        * 100
    )

    print(
        f"Percentage removed    : "
        f"{removal_percentage:.4f}%"
    )

    # --------------------------------------------------------
    # Check class distribution of removed observations
    # --------------------------------------------------------
    if len(removed_records) > 0:

        print(
            "\nTarget distribution among removed records:"
        )

        display(
            removed_records["Dry_Eye"]
            .value_counts()
            .sort_index()
            .rename(index={
                0: "No Dry Eye",
                1: "Dry Eye"
            })
            .to_frame(name="Count")
        )

    # --------------------------------------------------------
    # Remaining target distribution
    # --------------------------------------------------------
    final_target_summary = pd.DataFrame({

        "Count":
            df_model["Dry_Eye"]
            .value_counts()
            .sort_index(),

        "Percentage":
            df_model["Dry_Eye"]
            .value_counts(normalize=True)
            .sort_index()
            .mul(100)
            .round(2)
    })

    final_target_summary.index = [
        "0 = No Dry Eye",
        "1 = Dry Eye"
    ]

    print("\nFinal target distribution:")

    display(final_target_summary)

    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------
    assert (
        df_model["Systolic BP"]
        >
        df_model["Diastolic BP"]
    ).all()

    assert df_model.isna().sum().sum() == 0

    print("\n✅ Invalid BP observations removed.")
    print("✅ Final dataset passed quality checks.")

except Exception as e:
    print("\n❌ STEP 7 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### Final Analysis Dataset

After data-quality filtering, the final analysis dataset is saved separately. This version will be used for exploratory analysis and all subsequent machine-learning experiments.

In [ ]:
# ============================================================
# STEP 8: SAVE FINAL ANALYSIS DATASET
# ============================================================

try:
    FINAL_DATA_PATH = (
        f"{OUT_DIR}/"
        "dry_eye_final_analysis_dataset.csv"
    )

    df_model.to_csv(
        FINAL_DATA_PATH,
        index=False
    )

    print("✅ Final analysis dataset saved.")
    print(f"Path: {FINAL_DATA_PATH}")
    print(f"Shape: {df_model.shape}")

except Exception as e:
    print("\n❌ STEP 8 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 7. Exploratory Data Analysis

Exploratory analysis is performed to understand the structure of the Dry Eye Disease dataset before predictive modeling.

The analysis focuses on:

- target-class distribution;
- demographic characteristics;
- sleep and health-related variables;
- lifestyle characteristics;
- digital-exposure variables;
- ocular symptoms.

The purpose of EDA is descriptive. Feature groups used later in the ablation experiment are defined according to the semantic meaning of the variables rather than selected based on their observed association with the target.

In [ ]:
# ============================================================
# STEP 9: TARGET DISTRIBUTION
# ============================================================

try:
    target_counts = (
        df_model["Dry_Eye"]
        .value_counts()
        .sort_index()
    )

    target_percentages = (
        df_model["Dry_Eye"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
    )

    target_eda = pd.DataFrame({
        "Count": target_counts,
        "Percentage": target_percentages.round(2)
    })

    target_eda.index = [
        "No Dry Eye",
        "Dry Eye"
    ]

    print("=" * 75)
    print("TARGET CLASS DISTRIBUTION")
    print("=" * 75)

    display(target_eda)

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------
    plt.figure(figsize=(7, 5))

    bars = plt.bar(
        target_eda.index,
        target_eda["Count"]
    )

    plt.title(
        "Distribution of Dry Eye Disease"
    )

    plt.xlabel("Class")
    plt.ylabel("Number of Observations")

    # Add percentage labels above bars
    for bar, percentage in zip(
        bars,
        target_eda["Percentage"]
    ):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{percentage:.1f}%",
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()

    print("✅ Target distribution visualized successfully.")

except Exception as e:
    print("\n❌ TARGET EDA FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 7.1 Numerical Feature Summary

Descriptive statistics are calculated for the continuous and ordinal variables to examine their central tendency, variability, and observed ranges.

This provides a general overview of the demographic, sleep, physiological, lifestyle, and digital-exposure characteristics represented in the dataset.

In [ ]:
# ============================================================
# STEP 10: NUMERICAL FEATURE SUMMARY
# ============================================================

try:
    # Variables that are naturally measured numerically.
    # Binary 0/1 indicators are intentionally excluded from
    # this descriptive table.
    continuous_features = [

        "Age",
        "Sleep duration",
        "Sleep quality",
        "Stress level",

        "Heart rate",
        "Daily steps",
        "Physical activity",

        "Height",
        "Weight",

        "Average screen time",

        "Systolic BP",
        "Diastolic BP"
    ]

    # --------------------------------------------------------
    # Verify all expected columns exist
    # --------------------------------------------------------
    missing_continuous = [
        col
        for col in continuous_features
        if col not in df_model.columns
    ]

    if missing_continuous:
        raise KeyError(
            f"Expected numerical features missing: "
            f"{missing_continuous}"
        )

    numeric_summary = (
        df_model[continuous_features]
        .describe()
        .T
    )

    numeric_summary = numeric_summary[
        [
            "count",
            "mean",
            "std",
            "min",
            "25%",
            "50%",
            "75%",
            "max"
        ]
    ].round(2)

    print("=" * 75)
    print("NUMERICAL FEATURE SUMMARY")
    print("=" * 75)

    display(numeric_summary)

    print(
        "\n✅ Numerical feature summary created successfully."
    )

except Exception as e:
    print("\n❌ NUMERICAL SUMMARY FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 7.2 Selected Numerical Distributions

Selected continuous predictors are compared between participants with and without Dry Eye Disease.

The variables shown are:

- Age
- Sleep duration
- Average screen time
- Physical activity

These plots are exploratory and are not used to select features for the machine-learning models.

In [ ]:
# ============================================================
# STEP 11: SELECTED NUMERICAL DISTRIBUTIONS
# ============================================================

try:
    # Human-readable class labels for plotting
    plot_df = df_model.copy()

    plot_df["Dry Eye Status"] = (
        plot_df["Dry_Eye"]
        .map({
            0: "No Dry Eye",
            1: "Dry Eye"
        })
    )

    features_to_plot = [
        "Age",
        "Sleep duration",
        "Average screen time",
        "Physical activity"
    ]

    for feature in features_to_plot:

        plt.figure(figsize=(7, 4))

        sns.boxplot(
            data=plot_df,
            x="Dry Eye Status",
            y=feature
        )

        plt.title(
            f"{feature} by Dry Eye Disease Status"
        )

        plt.xlabel("")
        plt.ylabel(feature)

        plt.tight_layout()
        plt.show()

    print(
        "✅ Selected numerical distributions "
        "created successfully."
    )

except Exception as e:
    print("\n❌ NUMERICAL DISTRIBUTION PLOTS FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 7.3 Ocular Symptom Analysis

Three direct ocular symptoms are available:

- Discomfort / eye strain
- Eye redness
- Itchiness / irritation

Their prevalence is compared between Dry Eye Disease classes.

This analysis is particularly important because the later ablation experiment will evaluate how much predictive performance remains when direct ocular symptoms are excluded.

In [ ]:
# ============================================================
# STEP 12: OCULAR SYMPTOM ANALYSIS
# ============================================================

try:
    ocular_features = [
        "Discomfort Eye-strain",
        "Redness in eye",
        "Itchiness/Irritation in eye"
    ]

    ocular_results = []

    for feature in ocular_features:

        # Percentage of each class having the symptom
        grouped = (
            df_model
            .groupby("Dry_Eye")[feature]
            .mean()
            .mul(100)
        )

        ocular_results.append({
            "Feature": feature,
            "No Dry Eye (%)": grouped.get(0, np.nan),
            "Dry Eye (%)": grouped.get(1, np.nan)
        })

    ocular_summary = (
        pd.DataFrame(ocular_results)
        .set_index("Feature")
        .round(2)
    )

    print("=" * 75)
    print("OCULAR SYMPTOM PREVALENCE")
    print("=" * 75)

    display(ocular_summary)

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------
    ocular_summary.plot(
        kind="bar",
        figsize=(10, 5)
    )

    plt.title(
        "Ocular Symptom Prevalence by Dry Eye Disease Status"
    )

    plt.ylabel(
        "Participants Reporting Symptom (%)"
    )

    plt.xlabel("")

    plt.xticks(
        rotation=20,
        ha="right"
    )

    plt.legend(
        title="Disease Status"
    )

    plt.tight_layout()
    plt.show()

    print(
        "✅ Ocular symptom analysis completed successfully."
    )

except Exception as e:
    print("\n❌ OCULAR SYMPTOM ANALYSIS FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 7.4 Sleep, Lifestyle, and Digital-Exposure Indicators

Selected behavioral and exposure-related indicators are compared by Dry Eye Disease status.

The purpose is to describe possible differences in sleep behavior, substance use, smart-device use, and blue-light-filter use before machine-learning analysis.

In [ ]:
# ============================================================
# STEP 13: BINARY BEHAVIORAL / EXPOSURE ANALYSIS
# ============================================================

try:
    selected_binary_features = [

        "Sleep disorder",
        "Wake up during night",
        "Feel sleepy during day",

        "Caffeine consumption",
        "Alcohol consumption",
        "Smoking",

        "Smart device before bed",
        "Blue-light filter"
    ]

    binary_results = []

    for feature in selected_binary_features:

        grouped = (
            df_model
            .groupby("Dry_Eye")[feature]
            .mean()
            .mul(100)
        )

        binary_results.append({
            "Feature": feature,
            "No Dry Eye (%)": grouped.get(0, np.nan),
            "Dry Eye (%)": grouped.get(1, np.nan)
        })

    behavioral_summary = (
        pd.DataFrame(binary_results)
        .set_index("Feature")
        .round(2)
    )

    print("=" * 75)
    print("BEHAVIORAL / EXPOSURE PREVALENCE")
    print("=" * 75)

    display(behavioral_summary)

    behavioral_summary.plot(
        kind="bar",
        figsize=(11, 6)
    )

    plt.title(
        "Behavioral and Exposure Indicators by Dry Eye Status"
    )

    plt.ylabel(
        "Participants with Indicator (%)"
    )

    plt.xlabel("")

    plt.xticks(
        rotation=35,
        ha="right"
    )

    plt.tight_layout()
    plt.show()

    print(
        "✅ Behavioral/exposure analysis completed."
    )

except Exception as e:
    print("\n❌ BEHAVIORAL ANALYSIS FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 7.5 Univariate Association with Dry Eye Disease

After preprocessing, all predictors are numerical. Their individual linear associations with the binary Dry Eye Disease outcome are therefore examined.

For continuous predictors, this corresponds to point-biserial correlation with the binary outcome.

For binary predictors, it represents the correlation between two binary variables.

These associations are used only for descriptive analysis and not for feature selection.

In [ ]:
# ============================================================
# STEP 14: UNIVARIATE ASSOCIATION WITH TARGET
# ============================================================

try:
    feature_columns = [
        col
        for col in df_model.columns
        if col != "Dry_Eye"
    ]

    target_correlations = (
        df_model[feature_columns + ["Dry_Eye"]]
        .corr()["Dry_Eye"]
        .drop("Dry_Eye")
    )

    association_table = pd.DataFrame({
        "Correlation": target_correlations,
        "Absolute_Correlation":
            target_correlations.abs()
    })

    association_table = (
        association_table
        .sort_values(
            "Absolute_Correlation",
            ascending=False
        )
    )

    print("=" * 75)
    print("FEATURE ASSOCIATIONS WITH DRY EYE")
    print("=" * 75)

    display(
        association_table
        .round(4)
    )

    # --------------------------------------------------------
    # Top 15 visualization
    # --------------------------------------------------------
    top_associations = (
        association_table
        .head(15)
        .sort_values(
            "Correlation"
        )
    )

    plt.figure(figsize=(9, 6))

    plt.barh(
        top_associations.index,
        top_associations["Correlation"]
    )

    plt.axvline(
        0,
        linewidth=1
    )

    plt.xlabel(
        "Correlation with Dry Eye Disease"
    )

    plt.title(
        "Top Univariate Associations with Dry Eye Disease"
    )

    plt.tight_layout()
    plt.show()

    print(
        "✅ Univariate association analysis completed."
    )

except Exception as e:
    print("\n❌ ASSOCIATION ANALYSIS FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 8. Feature Group Definition

To investigate the source of predictive information, the predictors are divided into five conceptually defined groups.

### Demographic
General personal and anthropometric characteristics.

### Sleep / Health
Sleep characteristics, physiological measures, and general health indicators.

### Lifestyle
Physical activity and substance-use behaviors.

### Digital Exposure
Variables directly related to digital-device exposure and screen behavior.

### Ocular Symptoms
Direct eye-related symptoms.

These groups are defined before predictive modeling and are not based on statistical feature importance. They will later be used in a controlled feature-group ablation study.

In [ ]:
# ============================================================
# STEP 15: DEFINE FEATURE GROUPS
# ============================================================

try:
    FEATURE_GROUPS = {

        # ----------------------------------------------------
        # Demographic / anthropometric
        # ----------------------------------------------------
        "Demographic": [
            "Gender_Male",
            "Age",
            "Height",
            "Weight"
        ],

        # ----------------------------------------------------
        # Sleep, physiology and health
        # ----------------------------------------------------
        "Sleep_Health": [
            "Sleep duration",
            "Sleep quality",
            "Stress level",

            "Systolic BP",
            "Diastolic BP",
            "Heart rate",

            "Sleep disorder",
            "Wake up during night",
            "Feel sleepy during day",

            "Medical issue",
            "Ongoing medication"
        ],

        # ----------------------------------------------------
        # Lifestyle behaviors
        # ----------------------------------------------------
        "Lifestyle": [
            "Daily steps",
            "Physical activity",

            "Caffeine consumption",
            "Alcohol consumption",
            "Smoking"
        ],

        # ----------------------------------------------------
        # Digital-device exposure
        # ----------------------------------------------------
        "Digital_Exposure": [
            "Smart device before bed",
            "Average screen time",
            "Blue-light filter"
        ],

        # ----------------------------------------------------
        # Direct ocular symptoms
        # ----------------------------------------------------
        "Ocular_Symptoms": [
            "Discomfort Eye-strain",
            "Redness in eye",
            "Itchiness/Irritation in eye"
        ]
    }

    print("=" * 75)
    print("FEATURE GROUP DEFINITIONS")
    print("=" * 75)

    # --------------------------------------------------------
    # Display groups
    # --------------------------------------------------------
    group_rows = []

    for group_name, features in FEATURE_GROUPS.items():

        print(
            f"\n{group_name} "
            f"({len(features)} features)"
        )

        for feature in features:
            print(f"  - {feature}")

            group_rows.append({
                "Group": group_name,
                "Feature": feature
            })

    feature_group_table = pd.DataFrame(
        group_rows
    )

    # --------------------------------------------------------
    # Verify coverage
    # --------------------------------------------------------
    grouped_features = [
        feature
        for features in FEATURE_GROUPS.values()
        for feature in features
    ]

    actual_predictors = [
        col
        for col in df_model.columns
        if col != "Dry_Eye"
    ]

    missing_from_groups = sorted(
        set(actual_predictors)
        -
        set(grouped_features)
    )

    extra_in_groups = sorted(
        set(grouped_features)
        -
        set(actual_predictors)
    )

    duplicated_group_features = (
        pd.Series(grouped_features)
        .value_counts()
    )

    duplicated_group_features = (
        duplicated_group_features[
            duplicated_group_features > 1
        ]
    )

    print("\n" + "-" * 75)

    print(
        f"Total predictors in dataset : "
        f"{len(actual_predictors)}"
    )

    print(
        f"Predictors assigned to groups: "
        f"{len(grouped_features)}"
    )

    print(
        f"Missing from groups         : "
        f"{missing_from_groups}"
    )

    print(
        f"Unexpected group features   : "
        f"{extra_in_groups}"
    )

    print(
        f"Duplicated between groups   : "
        f"{duplicated_group_features.to_dict()}"
    )

    # --------------------------------------------------------
    # Assertions
    # --------------------------------------------------------
    if missing_from_groups:
        raise ValueError(
            "Some predictors were not assigned to a group."
        )

    if extra_in_groups:
        raise ValueError(
            "A feature group contains nonexistent predictors."
        )

    if len(duplicated_group_features) > 0:
        raise ValueError(
            "Some predictors were assigned to multiple groups."
        )

    print("\n✅ Every predictor belongs to exactly one group.")

except Exception as e:
    print("\n❌ FEATURE GROUP DEFINITION FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 8.1 Pre-Specified Feature-Group Experiments

Five main feature configurations are defined before model training:

1. **All Features** — complete predictor set.
2. **No Ocular Symptoms** — all predictors except direct ocular symptoms.
3. **Lifestyle + Digital Only** — behavioral and digital-exposure predictors.
4. **Ocular Symptoms Only** — direct eye symptoms only.
5. **Incremental Groups** — feature groups are progressively combined to examine where predictive information is gained.

This design allows the study to distinguish general risk-factor prediction from symptom-driven classification.

In [ ]:
# ============================================================
# STEP 16: PRE-SPECIFY ABLATION CONFIGURATIONS
# ============================================================

try:
    # --------------------------------------------------------
    # Convenient aliases
    # --------------------------------------------------------
    demographic = FEATURE_GROUPS["Demographic"]
    sleep_health = FEATURE_GROUPS["Sleep_Health"]
    lifestyle = FEATURE_GROUPS["Lifestyle"]
    digital = FEATURE_GROUPS["Digital_Exposure"]
    ocular = FEATURE_GROUPS["Ocular_Symptoms"]

    all_features = (
        demographic
        + sleep_health
        + lifestyle
        + digital
        + ocular
    )

    no_ocular_features = [
        feature
        for feature in all_features
        if feature not in ocular
    ]

    lifestyle_digital_features = (
        lifestyle
        + digital
    )

    ocular_only_features = ocular.copy()

    # --------------------------------------------------------
    # Main ablation configurations
    # --------------------------------------------------------
    ABLATION_CONFIGS = {

        "All Features":
            all_features,

        "No Ocular Symptoms":
            no_ocular_features,

        "Lifestyle + Digital Only":
            lifestyle_digital_features,

        "Ocular Symptoms Only":
            ocular_only_features
    }

    # --------------------------------------------------------
    # Incremental configurations
    #
    # This examines performance gained as additional
    # information groups are introduced.
    # --------------------------------------------------------
    INCREMENTAL_CONFIGS = {

        "1. Demographic":
            demographic,

        "2. + Sleep/Health":
            demographic
            + sleep_health,

        "3. + Lifestyle":
            demographic
            + sleep_health
            + lifestyle,

        "4. + Digital Exposure":
            demographic
            + sleep_health
            + lifestyle
            + digital,

        "5. + Ocular Symptoms":
            all_features
    }

    print("=" * 75)
    print("PRE-SPECIFIED MODELING CONFIGURATIONS")
    print("=" * 75)

    print("\nMain ablation experiments:")

    for name, features in ABLATION_CONFIGS.items():
        print(
            f"{name:<28} : "
            f"{len(features)} features"
        )

    print("\nIncremental experiments:")

    for name, features in INCREMENTAL_CONFIGS.items():
        print(
            f"{name:<28} : "
            f"{len(features)} features"
        )

    print(
        "\n✅ Ablation configurations created successfully."
    )

except Exception as e:
    print("\n❌ ABLATION CONFIGURATION FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 9. Train-Test Partition

The final dataset is divided using stratified sampling:

- **80% training data**
- **20% held-out test data**

Stratification preserves the Dry Eye Disease class distribution.

The test set is locked and will not be used for model selection, feature-group selection, or hyperparameter decisions. Cross-validation on the training data will be used for model comparison.

In [ ]:
# ============================================================
# STEP 17: CREATE LOCKED TRAIN / TEST SPLIT
# CHECKPOINT 2
# ============================================================

try:
    from sklearn.model_selection import train_test_split

    TARGET = "Dry_Eye"

    # --------------------------------------------------------
    # Feature matrix and target
    # --------------------------------------------------------
    X = df_model.drop(
        columns=[TARGET]
    ).copy()

    y = df_model[TARGET].copy()

    # --------------------------------------------------------
    # Stratified 80/20 split
    # --------------------------------------------------------
    X_train, X_test, y_train, y_test = (
        train_test_split(
            X,
            y,

            test_size=0.20,

            random_state=RANDOM_STATE,

            stratify=y
        )
    )

    print("=" * 75)
    print("LOCKED TRAIN / TEST SPLIT")
    print("=" * 75)

    print(f"\nFull dataset : {X.shape}")
    print(f"Training set : {X_train.shape}")
    print(f"Test set     : {X_test.shape}")

    # --------------------------------------------------------
    # Helper function
    # --------------------------------------------------------
    def target_distribution(series):

        counts = (
            series
            .value_counts()
            .sort_index()
        )

        percentages = (
            series
            .value_counts(normalize=True)
            .sort_index()
            .mul(100)
            .round(2)
        )

        result = pd.DataFrame({
            "Count": counts,
            "Percentage": percentages
        })

        result.index = [
            "No Dry Eye",
            "Dry Eye"
        ]

        return result

    print("\nTraining class distribution:")
    display(
        target_distribution(y_train)
    )

    print("\nTest class distribution:")
    display(
        target_distribution(y_test)
    )

    # --------------------------------------------------------
    # Ensure there is no row overlap
    # --------------------------------------------------------
    overlap = (
        set(X_train.index)
        .intersection(
            set(X_test.index)
        )
    )

    print(
        f"\nTrain/Test index overlap: "
        f"{len(overlap)}"
    )

    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------
    if len(overlap) != 0:
        raise ValueError(
            "Train/test overlap detected."
        )

    if TARGET in X_train.columns:
        raise ValueError(
            "Target leakage detected in X_train."
        )

    if TARGET in X_test.columns:
        raise ValueError(
            "Target leakage detected in X_test."
        )

    if (
        len(X_train)
        +
        len(X_test)
        !=
        len(df_model)
    ):
        raise ValueError(
            "Train/test row counts do not match dataset."
        )

    print("\n✅ Stratified split completed.")
    print("✅ No train/test overlap detected.")
    print("✅ No target leakage detected.")
    print("✅ Test set is now LOCKED.")

    print("\n" + "=" * 75)
    print("✅ CHECKPOINT 2 PASSED.")
    print("=" * 75)

except Exception as e:
    print("\n❌ CHECKPOINT 2 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### Reproducibility of the Data Split

The row indices used for the training and held-out test sets are saved so that the same partition can be reproduced later.

In [ ]:
# ============================================================
# STEP 18: SAVE TRAIN / TEST INDICES
# ============================================================

try:
    split_record = pd.DataFrame({

        "Original_Index":
            list(X_train.index)
            +
            list(X_test.index),

        "Dataset_Split":
            ["Train"] * len(X_train)
            +
            ["Test"] * len(X_test)
    })

    split_path = (
        f"{OUT_DIR}/"
        "dry_eye_train_test_split.csv"
    )

    split_record.to_csv(
        split_path,
        index=False
    )

    print("✅ Train/test split record saved.")
    print(f"Path: {split_path}")

except Exception as e:
    print("\n❌ SPLIT SAVE FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## Checkpoint 2 Summary

This cell collects the key outputs from EDA, feature grouping, and the locked train/test split into one place before model training begins.

In [ ]:
# ============================================================
# CHECKPOINT 2 SUMMARY
# Run this before starting machine learning
# ============================================================

try:
    print("=" * 80)
    print("CHECKPOINT 2 SUMMARY")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Final dataset size
    # --------------------------------------------------------
    print("\n1. FINAL DATASET SIZE")
    print("-" * 80)

    print(f"Observations : {len(df_model):,}")
    print(f"Predictors   : {df_model.shape[1] - 1}")
    print(f"Target       : Dry_Eye")

    # --------------------------------------------------------
    # 2. Ocular symptom prevalence
    # --------------------------------------------------------
    print("\n2. OCULAR SYMPTOM PREVALENCE")
    print("-" * 80)

    display(ocular_summary)

    # --------------------------------------------------------
    # 3. Top 10 target associations
    # --------------------------------------------------------
    print("\n3. TOP 10 FEATURES ASSOCIATED WITH DRY EYE")
    print("-" * 80)

    display(
        association_table
        .head(10)
        .round(4)
    )

    # --------------------------------------------------------
    # 4. Feature group sizes
    # --------------------------------------------------------
    print("\n4. FEATURE GROUP SIZES")
    print("-" * 80)

    group_summary = pd.DataFrame({
        "Feature_Group": list(FEATURE_GROUPS.keys()),
        "Number_of_Features": [
            len(features)
            for features in FEATURE_GROUPS.values()
        ]
    })

    display(group_summary)

    # --------------------------------------------------------
    # 5. Main ablation configurations
    # --------------------------------------------------------
    print("\n5. MAIN ABLATION CONFIGURATIONS")
    print("-" * 80)

    ablation_summary = pd.DataFrame({
        "Configuration": list(ABLATION_CONFIGS.keys()),
        "Number_of_Features": [
            len(features)
            for features in ABLATION_CONFIGS.values()
        ]
    })

    display(ablation_summary)

    # --------------------------------------------------------
    # 6. Train/test sizes
    # --------------------------------------------------------
    print("\n6. TRAIN / TEST SPLIT")
    print("-" * 80)

    print(f"Training set : {X_train.shape}")
    print(f"Test set     : {X_test.shape}")

    # --------------------------------------------------------
    # 7. Class distributions
    # --------------------------------------------------------
    print("\nTraining target distribution:")
    display(target_distribution(y_train))

    print("\nTest target distribution:")
    display(target_distribution(y_test))

    # --------------------------------------------------------
    # 8. Integrity checks
    # --------------------------------------------------------
    overlap = set(X_train.index).intersection(
        set(X_test.index)
    )

    if len(overlap) != 0:
        raise ValueError(
            f"Train/test overlap detected: {len(overlap)} rows."
        )

    if "Dry_Eye" in X_train.columns:
        raise ValueError(
            "Target leakage detected in training features."
        )

    print("\n✅ No train/test overlap.")
    print("✅ No target leakage.")
    print("✅ Feature groups available.")
    print("✅ Ablation configurations available.")

    print("\n" + "=" * 80)
    print("✅ CHECKPOINT 2 SUMMARY COMPLETED SUCCESSFULLY.")
    print("=" * 80)

except Exception as e:
    print("\n❌ CHECKPOINT 2 SUMMARY FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 10. Baseline Machine-Learning Models

Before conducting the feature-group ablation study, several conventional machine-learning algorithms are evaluated using the complete feature set.

The models are:

1. Dummy Classifier — no-skill reference
2. Logistic Regression — linear baseline
3. Random Forest — tree-based ensemble
4. XGBoost — gradient-boosting ensemble

Model evaluation is performed using **stratified 5-fold cross-validation on the training data only**.

The held-out test set remains untouched.

Because the classes are moderately imbalanced, model performance is evaluated using multiple metrics:

- Accuracy
- Balanced Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- PR-AUC

Accuracy alone is not considered sufficient because a majority-class classifier can already obtain relatively high accuracy.

In [ ]:
# ============================================================
# STEP 20: CHECK REQUIRED MODEL LIBRARIES
# ============================================================

try:
    import xgboost as xgb

    print("=" * 70)
    print("MODEL LIBRARY CHECK")
    print("=" * 70)

    print(f"\nXGBoost version: {xgb.__version__}")
    print("✅ XGBoost is available.")

except ImportError as e:

    print("❌ XGBoost is not currently available.")
    print("Error:", str(e))

    print(
        "\nIf this happens, run the following "
        "in a NEW code cell:"
    )

    print("pip install -q xgboost")
    print("\nThen rerun this cell.")

### 10.1 Cross-Validation Strategy

A stratified 5-fold cross-validation procedure is used.

Stratification ensures that each fold contains approximately the same proportion of Dry Eye and non-Dry-Eye observations.

The same folds are used for every model so that later statistical comparisons are paired and fair.

In [ ]:
# ============================================================
# STEP 21: CROSS-VALIDATION AND EVALUATION METRICS
# ============================================================

try:
    from sklearn.model_selection import StratifiedKFold

    # --------------------------------------------------------
    # Five stratified folds
    # --------------------------------------------------------
    cv5 = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    # --------------------------------------------------------
    # Metrics used for every baseline model
    # --------------------------------------------------------
    scoring_metrics = {
        "Accuracy": "accuracy",
        "Balanced_Accuracy": "balanced_accuracy",
        "Precision": "precision",
        "Recall": "recall",
        "F1": "f1",
        "ROC_AUC": "roc_auc",
        "PR_AUC": "average_precision"
    }

    print("=" * 70)
    print("CROSS-VALIDATION SETUP")
    print("=" * 70)

    print("\nNumber of folds : 5")
    print("Stratified      : Yes")
    print("Shuffle         : Yes")
    print(f"Random state    : {RANDOM_STATE}")

    print("\nEvaluation metrics:")

    for metric in scoring_metrics:
        print(f"- {metric}")

    print("\n✅ Cross-validation configuration ready.")

except Exception as e:
    print("\n❌ CROSS-VALIDATION SETUP FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 10.2 Baseline Model Configuration

Logistic Regression is combined with standardization because its coefficients are sensitive to predictor scale.

Random Forest and XGBoost operate directly on the numerical feature values and therefore do not require standardization.

The models use fixed random seeds to improve reproducibility.

In [ ]:
# ============================================================
# STEP 22: DEFINE BASELINE MODELS
# ============================================================

try:
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    from sklearn.dummy import DummyClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier

    from xgboost import XGBClassifier

    # --------------------------------------------------------
    # Dummy model
    #
    # Predicts according to the observed class prior.
    # It provides a no-skill reference.
    # --------------------------------------------------------
    dummy_model = DummyClassifier(
        strategy="prior"
    )

    # --------------------------------------------------------
    # Logistic Regression
    #
    # StandardScaler is fitted inside each CV fold.
    # class_weight='balanced' reduces majority-class bias.
    # --------------------------------------------------------
    logistic_model = Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler()
            ),
            (
                "classifier",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE
                )
            )
        ]
    )

    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------
    random_forest_model = RandomForestClassifier(
        n_estimators=250,

        class_weight="balanced",

        random_state=RANDOM_STATE,

        n_jobs=-1
    )

    # --------------------------------------------------------
    # XGBoost
    #
    # tree_method='hist' is efficient for CPU tabular data.
    # --------------------------------------------------------
    xgboost_model = XGBClassifier(
        n_estimators=250,

        learning_rate=0.05,

        max_depth=4,

        min_child_weight=1,

        subsample=0.85,

        colsample_bytree=0.85,

        objective="binary:logistic",

        eval_metric="logloss",

        tree_method="hist",

        random_state=RANDOM_STATE,

        n_jobs=-1,

        verbosity=0
    )

    # --------------------------------------------------------
    # Store models in one dictionary
    # --------------------------------------------------------
    BASELINE_MODELS = {

        "Dummy Baseline":
            dummy_model,

        "Logistic Regression":
            logistic_model,

        "Random Forest":
            random_forest_model,

        "XGBoost":
            xgboost_model
    }

    print("=" * 70)
    print("BASELINE MODELS")
    print("=" * 70)

    for model_name in BASELINE_MODELS:
        print(f"✅ {model_name}")

    print(
        "\n✅ All baseline models created successfully."
    )

except Exception as e:
    print("\n❌ MODEL CREATION FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 10.3 Baseline Cross-Validation

Each baseline model is evaluated using the same five training folds.

Both the mean performance and the variation across folds are retained.

The fold-level results are also stored for later statistical comparison.

In [ ]:
# ============================================================
# STEP 23: BASELINE MODEL CROSS-VALIDATION
# CHECKPOINT 3A
# ============================================================

try:
    import time

    from sklearn.model_selection import cross_validate

    print("=" * 80)
    print("BASELINE MODEL CROSS-VALIDATION")
    print("=" * 80)

    baseline_summary_rows = []

    baseline_fold_rows = []

    # --------------------------------------------------------
    # Evaluate each model using exactly the same CV object
    # --------------------------------------------------------
    for model_name, model in BASELINE_MODELS.items():

        print(f"\nRunning: {model_name}")

        start_time = time.time()

        cv_results = cross_validate(
            estimator=model,

            X=X_train,
            y=y_train,

            cv=cv5,

            scoring=scoring_metrics,

            # Sequential outer CV avoids excessive
            # parallelism because RF/XGB already use CPU cores
            n_jobs=1,

            return_train_score=False,

            error_score="raise"
        )

        elapsed_time = (
            time.time() - start_time
        )

        # ----------------------------------------------------
        # Save mean and standard deviation for each metric
        # ----------------------------------------------------
        summary_row = {
            "Model": model_name,
            "Time_seconds": elapsed_time
        }

        for metric in scoring_metrics:

            values = cv_results[
                f"test_{metric}"
            ]

            summary_row[
                f"{metric}_Mean"
            ] = np.mean(values)

            summary_row[
                f"{metric}_STD"
            ] = np.std(values)

            # Save every fold separately
            for fold_number, value in enumerate(
                values,
                start=1
            ):

                baseline_fold_rows.append({
                    "Model": model_name,
                    "Fold": fold_number,
                    "Metric": metric,
                    "Score": value
                })

        baseline_summary_rows.append(
            summary_row
        )

        print(
            f"✅ Finished in "
            f"{elapsed_time:.2f} seconds."
        )

    # --------------------------------------------------------
    # Create dataframes
    # --------------------------------------------------------
    baseline_results = pd.DataFrame(
        baseline_summary_rows
    )

    baseline_fold_results = pd.DataFrame(
        baseline_fold_rows
    )

    # --------------------------------------------------------
    # Sort models by ROC-AUC
    # --------------------------------------------------------
    baseline_results = (
        baseline_results
        .sort_values(
            "ROC_AUC_Mean",
            ascending=False
        )
        .reset_index(drop=True)
    )

    print("\n" + "=" * 80)
    print("BASELINE CROSS-VALIDATION RESULTS")
    print("=" * 80)

    # Main display table
    display_columns = [
        "Model",
        "Accuracy_Mean",
        "Balanced_Accuracy_Mean",
        "Precision_Mean",
        "Recall_Mean",
        "F1_Mean",
        "ROC_AUC_Mean",
        "PR_AUC_Mean",
        "Time_seconds"
    ]

    display(
        baseline_results[
            display_columns
        ].round(4)
    )

    print("\n✅ CHECKPOINT 3A COMPLETED SUCCESSFULLY.")

except Exception as e:
    print("\n❌ CHECKPOINT 3A FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

    print(
        "\nPlease send me this complete error message "
        "before changing the code."
    )

### 10.4 Baseline Performance Summary

Cross-validation results are summarized as mean ± standard deviation across the five folds.

Low standard deviation indicates that performance is relatively stable across different training subsets.

In [ ]:
# ============================================================
# STEP 24: CREATE MEAN +/- STANDARD DEVIATION TABLE
# ============================================================

try:
    paper_metrics = [
        "Accuracy",
        "Balanced_Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC",
        "PR_AUC"
    ]

    paper_rows = []

    for _, row in baseline_results.iterrows():

        result_row = {
            "Model": row["Model"]
        }

        for metric in paper_metrics:

            mean_value = row[
                f"{metric}_Mean"
            ]

            std_value = row[
                f"{metric}_STD"
            ]

            result_row[metric] = (
                f"{mean_value:.3f} "
                f"± {std_value:.3f}"
            )

        paper_rows.append(
            result_row
        )

    baseline_paper_table = pd.DataFrame(
        paper_rows
    )

    print("=" * 80)
    print("BASELINE PERFORMANCE: MEAN ± STD")
    print("=" * 80)

    display(baseline_paper_table)

    print(
        "\n✅ Publication-style baseline table created."
    )

except Exception as e:
    print("\n❌ TABLE CREATION FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 10.5 Baseline Interpretation

The best machine-learning model is compared with the no-skill Dummy Classifier.

Particular attention is given to ROC-AUC and balanced accuracy because these metrics are less misleading than raw accuracy under class imbalance.

In [ ]:
# ============================================================
# STEP 25: AUTOMATIC BASELINE INTERPRETATION
# ============================================================

try:
    # --------------------------------------------------------
    # Remove dummy model when identifying best ML model
    # --------------------------------------------------------
    real_models = baseline_results[
        baseline_results["Model"]
        !=
        "Dummy Baseline"
    ]

    # --------------------------------------------------------
    # Best model by ROC-AUC
    # --------------------------------------------------------
    best_model_row = (
        real_models
        .sort_values(
            "ROC_AUC_Mean",
            ascending=False
        )
        .iloc[0]
    )

    dummy_row = (
        baseline_results[
            baseline_results["Model"]
            ==
            "Dummy Baseline"
        ]
        .iloc[0]
    )

    print("=" * 75)
    print("BASELINE INTERPRETATION")
    print("=" * 75)

    print(
        f"\nBest model by ROC-AUC : "
        f"{best_model_row['Model']}"
    )

    print(
        f"ROC-AUC                : "
        f"{best_model_row['ROC_AUC_Mean']:.4f}"
    )

    print(
        f"Balanced Accuracy      : "
        f"{best_model_row['Balanced_Accuracy_Mean']:.4f}"
    )

    print(
        f"F1 Score               : "
        f"{best_model_row['F1_Mean']:.4f}"
    )

    print("\nDummy reference:")

    print(
        f"Dummy Accuracy         : "
        f"{dummy_row['Accuracy_Mean']:.4f}"
    )

    print(
        f"Dummy Balanced Accuracy: "
        f"{dummy_row['Balanced_Accuracy_Mean']:.4f}"
    )

    print(
        f"Dummy ROC-AUC          : "
        f"{dummy_row['ROC_AUC_Mean']:.4f}"
    )

    # --------------------------------------------------------
    # Simple diagnostic—not a publication conclusion yet
    # --------------------------------------------------------
    best_auc = best_model_row[
        "ROC_AUC_Mean"
    ]

    if best_auc < 0.55:

        print(
            "\n⚠️ Predictive discrimination is very weak."
        )

    elif best_auc < 0.65:

        print(
            "\nℹ️ Predictive discrimination is modest."
        )

    elif best_auc < 0.75:

        print(
            "\n✅ Predictive discrimination is moderate."
        )

    else:

        print(
            "\n✅ Predictive discrimination is relatively strong."
        )

    print(
        "\nThe feature-group ablation experiment "
        "will determine where this predictive signal comes from."
    )

except Exception as e:
    print("\n❌ INTERPRETATION FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 26: SAVE BASELINE RESULTS
# ============================================================

try:
    baseline_results.to_csv(
        f"{OUT_DIR}/"
        "baseline_cv_summary.csv",
        index=False
    )

    baseline_fold_results.to_csv(
        f"{OUT_DIR}/"
        "baseline_cv_fold_scores.csv",
        index=False
    )

    baseline_paper_table.to_csv(
        f"{OUT_DIR}/"
        "baseline_paper_table.csv",
        index=False
    )

    print("✅ Baseline results saved successfully.")

    print(
        "\nFiles created:"
    )

    print(
        "- baseline_cv_summary.csv"
    )

    print(
        "- baseline_cv_fold_scores.csv"
    )

    print(
        "- baseline_paper_table.csv"
    )

except Exception as e:
    print("\n❌ SAVING BASELINE RESULTS FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## Does Dry Eye classification genuinely benefit from demographic, sleep, lifestyle, and digital-exposure factors, or is model performance mainly driven by direct ocular symptoms?

## 11. Feature-Group Ablation Study

The central experiment of this study evaluates how different groups of predictors contribute to Dry Eye Disease classification.

Two representative models are used:

- Logistic Regression — best baseline ROC-AUC
- XGBoost — best baseline balanced accuracy

Four feature configurations are evaluated:

1. **All Features**
2. **No Ocular Symptoms**
3. **Lifestyle + Digital Only**
4. **Ocular Symptoms Only**

All experiments use the same stratified 5-fold cross-validation procedure on the training dataset.

The held-out test set remains untouched.

The primary metric is ROC-AUC, supported by PR-AUC and Balanced Accuracy.

In [ ]:
# ============================================================
# STEP 27: MAIN FEATURE-GROUP ABLATION EXPERIMENT
# ============================================================

try:
    import time
    import numpy as np
    import pandas as pd

    from sklearn.model_selection import cross_validate

    print("=" * 85)
    print("FEATURE-GROUP ABLATION STUDY")
    print("=" * 85)

    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------
    required_objects = [
        "X_train",
        "y_train",
        "cv5",
        "scoring_metrics",
        "ABLATION_CONFIGS",
        "logistic_model",
        "xgboost_model"
    ]

    missing_objects = [
        obj for obj in required_objects
        if obj not in globals()
    ]

    if missing_objects:
        raise NameError(
            f"Missing required notebook objects: {missing_objects}"
        )

    # --------------------------------------------------------
    # Models selected for the ablation experiment
    # --------------------------------------------------------
    ABLATION_MODELS = {
        "Logistic Regression": logistic_model,
        "XGBoost": xgboost_model
    }

    ablation_summary_rows = []
    ablation_fold_rows = []

    # --------------------------------------------------------
    # Run every model on every feature configuration
    # --------------------------------------------------------
    for config_name, feature_list in ABLATION_CONFIGS.items():

        print("\n" + "-" * 85)
        print(
            f"Configuration: {config_name} "
            f"({len(feature_list)} features)"
        )
        print("-" * 85)

        # Make sure every requested feature exists
        missing_features = [
            feature for feature in feature_list
            if feature not in X_train.columns
        ]

        if missing_features:
            raise KeyError(
                f"Missing features in '{config_name}': "
                f"{missing_features}"
            )

        X_config = X_train[feature_list].copy()

        for model_name, model in ABLATION_MODELS.items():

            print(f"Running {model_name}...", end=" ")

            start_time = time.time()

            cv_result = cross_validate(
                estimator=model,
                X=X_config,
                y=y_train,
                cv=cv5,
                scoring=scoring_metrics,
                n_jobs=1,
                return_train_score=False,
                error_score="raise"
            )

            elapsed = time.time() - start_time

            summary_row = {
                "Configuration": config_name,
                "Model": model_name,
                "N_Features": len(feature_list),
                "Time_seconds": elapsed
            }

            # -----------------------------------------------
            # Store mean + standard deviation
            # -----------------------------------------------
            for metric in scoring_metrics:

                fold_values = cv_result[
                    f"test_{metric}"
                ]

                summary_row[
                    f"{metric}_Mean"
                ] = np.mean(fold_values)

                summary_row[
                    f"{metric}_STD"
                ] = np.std(fold_values)

                # Store each individual fold result
                for fold_number, score in enumerate(
                    fold_values,
                    start=1
                ):
                    ablation_fold_rows.append({
                        "Configuration": config_name,
                        "Model": model_name,
                        "Fold": fold_number,
                        "Metric": metric,
                        "Score": score
                    })

            ablation_summary_rows.append(
                summary_row
            )

            print(
                f"✅ {elapsed:.2f} sec"
            )

    # --------------------------------------------------------
    # Convert to dataframes
    # --------------------------------------------------------
    ablation_results = pd.DataFrame(
        ablation_summary_rows
    )

    ablation_fold_results = pd.DataFrame(
        ablation_fold_rows
    )

    # Sort for easier reading
    ablation_results = (
        ablation_results
        .sort_values(
            ["Model", "ROC_AUC_Mean"],
            ascending=[True, False]
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Display main result table
    # --------------------------------------------------------
    display_columns = [
        "Configuration",
        "Model",
        "N_Features",
        "Accuracy_Mean",
        "Balanced_Accuracy_Mean",
        "Precision_Mean",
        "Recall_Mean",
        "F1_Mean",
        "ROC_AUC_Mean",
        "PR_AUC_Mean"
    ]

    print("\n" + "=" * 85)
    print("MAIN ABLATION RESULTS")
    print("=" * 85)

    display(
        ablation_results[
            display_columns
        ].round(4)
    )

    print("\n✅ STEP 27 COMPLETED SUCCESSFULLY.")

except Exception as e:
    print("\n❌ STEP 27 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 11.1 Performance Change Relative to the Full Feature Set

To quantify the contribution of each feature group, every ablation configuration is compared against the corresponding model trained using all predictors.

The change in ROC-AUC is calculated as:

**Δ ROC-AUC = Ablation ROC-AUC − Full-Feature ROC-AUC**

A large negative value indicates that removing information substantially reduces discrimination.

In [ ]:
# ============================================================
# STEP 28: PERFORMANCE CHANGE RELATIVE TO ALL FEATURES
# ============================================================

try:
    comparison_rows = []

    for model_name in ABLATION_MODELS.keys():

        model_results = ablation_results[
            ablation_results["Model"] == model_name
        ].copy()

        # -----------------------------------------------
        # Full-feature reference
        # -----------------------------------------------
        full_row = model_results[
            model_results["Configuration"]
            == "All Features"
        ]

        if len(full_row) != 1:
            raise ValueError(
                f"Could not uniquely identify the "
                f"All Features row for {model_name}."
            )

        full_row = full_row.iloc[0]

        for _, row in model_results.iterrows():

            comparison_rows.append({

                "Model":
                    model_name,

                "Configuration":
                    row["Configuration"],

                "N_Features":
                    row["N_Features"],

                "ROC_AUC":
                    row["ROC_AUC_Mean"],

                "Delta_ROC_AUC":
                    row["ROC_AUC_Mean"]
                    -
                    full_row["ROC_AUC_Mean"],

                "PR_AUC":
                    row["PR_AUC_Mean"],

                "Delta_PR_AUC":
                    row["PR_AUC_Mean"]
                    -
                    full_row["PR_AUC_Mean"],

                "Balanced_Accuracy":
                    row["Balanced_Accuracy_Mean"],

                "Delta_Balanced_Accuracy":
                    row["Balanced_Accuracy_Mean"]
                    -
                    full_row[
                        "Balanced_Accuracy_Mean"
                    ]
            })

    ablation_delta_results = pd.DataFrame(
        comparison_rows
    )

    print("=" * 85)
    print("CHANGE RELATIVE TO ALL FEATURES")
    print("=" * 85)

    display(
        ablation_delta_results.round(4)
    )

    print(
        "\nInterpretation:"
    )

    print(
        "• Negative ΔROC-AUC = worse than full feature model"
    )

    print(
        "• Near-zero ΔROC-AUC = similar discrimination"
    )

    print(
        "• Positive ΔROC-AUC = ablation performed better"
    )

    print("\n✅ Relative performance table created.")

except Exception as e:
    print("\n❌ STEP 28 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 11.2 ROC-AUC Across Feature Configurations

ROC-AUC is visualized across the predefined feature configurations.

This figure directly shows whether direct ocular symptoms contribute most of the discrimination achieved by the classifiers.

In [ ]:
# ============================================================
# STEP 29: ABLATION ROC-AUC VISUALIZATION
# ============================================================

try:
    import matplotlib.pyplot as plt

    plot_data = ablation_results.copy()

    # --------------------------------------------------------
    # Create one figure per model for easier interpretation
    # --------------------------------------------------------
    for model_name in ABLATION_MODELS.keys():

        model_plot = (
            plot_data[
                plot_data["Model"] == model_name
            ]
            .copy()
        )

        # Fixed logical ordering
        desired_order = [
            "All Features",
            "No Ocular Symptoms",
            "Lifestyle + Digital Only",
            "Ocular Symptoms Only"
        ]

        model_plot["Configuration"] = pd.Categorical(
            model_plot["Configuration"],
            categories=desired_order,
            ordered=True
        )

        model_plot = model_plot.sort_values(
            "Configuration"
        )

        plt.figure(figsize=(9, 5))

        bars = plt.bar(
            model_plot["Configuration"].astype(str),
            model_plot["ROC_AUC_Mean"]
        )

        # Chance reference
        plt.axhline(
            0.50,
            linestyle="--",
            linewidth=1.5,
            label="Chance ROC-AUC = 0.50"
        )

        # Add values over bars
        for bar, auc in zip(
            bars,
            model_plot["ROC_AUC_Mean"]
        ):
            plt.text(
                bar.get_x()
                +
                bar.get_width() / 2,
                bar.get_height() + 0.003,
                f"{auc:.3f}",
                ha="center"
            )

        plt.ylim(
            0.45,
            max(
                0.65,
                model_plot["ROC_AUC_Mean"].max()
                + 0.04
            )
        )

        plt.ylabel("Mean 5-Fold ROC-AUC")

        plt.title(
            f"Feature-Group Ablation — {model_name}"
        )

        plt.xticks(
            rotation=20,
            ha="right"
        )

        plt.legend()

        plt.tight_layout()
        plt.show()

    print(
        "✅ Ablation ROC-AUC figures created successfully."
    )

except Exception as e:
    print("\n❌ STEP 29 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 12. Incremental Feature-Group Analysis

A second experiment progressively adds predictor groups to determine where measurable predictive information enters the model.

The sequence is:

1. Demographic
2. + Sleep / Health
3. + Lifestyle
4. + Digital Exposure
5. + Ocular Symptoms

If the addition of ocular symptoms produces a substantially larger increase in ROC-AUC than earlier groups, this would indicate that classification performance is predominantly symptom dependent.

In [ ]:
# ============================================================
# STEP 30: INCREMENTAL FEATURE-GROUP EXPERIMENT
# ============================================================

try:
    import time
    from sklearn.model_selection import cross_validate

    incremental_summary_rows = []
    incremental_fold_rows = []

    print("=" * 85)
    print("INCREMENTAL FEATURE-GROUP ANALYSIS")
    print("=" * 85)

    for config_name, feature_list in (
        INCREMENTAL_CONFIGS.items()
    ):

        print("\n" + "-" * 85)

        print(
            f"{config_name} "
            f"({len(feature_list)} features)"
        )

        print("-" * 85)

        X_config = X_train[
            feature_list
        ].copy()

        for model_name, model in (
            ABLATION_MODELS.items()
        ):

            print(
                f"Running {model_name}...",
                end=" "
            )

            start_time = time.time()

            cv_result = cross_validate(
                estimator=model,
                X=X_config,
                y=y_train,
                cv=cv5,
                scoring=scoring_metrics,
                n_jobs=1,
                return_train_score=False,
                error_score="raise"
            )

            elapsed = (
                time.time()
                -
                start_time
            )

            row = {
                "Configuration":
                    config_name,

                "Model":
                    model_name,

                "N_Features":
                    len(feature_list),

                "Time_seconds":
                    elapsed
            }

            for metric in scoring_metrics:

                fold_values = cv_result[
                    f"test_{metric}"
                ]

                row[
                    f"{metric}_Mean"
                ] = np.mean(
                    fold_values
                )

                row[
                    f"{metric}_STD"
                ] = np.std(
                    fold_values
                )

                for fold_number, score in enumerate(
                    fold_values,
                    start=1
                ):

                    incremental_fold_rows.append({

                        "Configuration":
                            config_name,

                        "Model":
                            model_name,

                        "Fold":
                            fold_number,

                        "Metric":
                            metric,

                        "Score":
                            score
                    })

            incremental_summary_rows.append(
                row
            )

            print(
                f"✅ {elapsed:.2f} sec"
            )

    incremental_results = pd.DataFrame(
        incremental_summary_rows
    )

    incremental_fold_results = pd.DataFrame(
        incremental_fold_rows
    )

    display_columns = [
        "Configuration",
        "Model",
        "N_Features",
        "Balanced_Accuracy_Mean",
        "F1_Mean",
        "ROC_AUC_Mean",
        "PR_AUC_Mean"
    ]

    print("\n" + "=" * 85)
    print("INCREMENTAL FEATURE RESULTS")
    print("=" * 85)

    display(
        incremental_results[
            display_columns
        ].round(4)
    )

    print("\n✅ STEP 30 COMPLETED SUCCESSFULLY.")

except Exception as e:
    print("\n❌ STEP 30 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 12.1 Incremental Gain in Predictive Discrimination

The change in ROC-AUC after adding each feature group is calculated.

This allows the contribution of demographic, health, lifestyle, digital, and ocular information to be compared directly.

In [ ]:
# ============================================================
# STEP 31: CALCULATE INCREMENTAL ROC-AUC GAINS
# ============================================================

try:
    incremental_gain_rows = []

    incremental_order = list(
        INCREMENTAL_CONFIGS.keys()
    )

    for model_name in ABLATION_MODELS.keys():

        model_inc = (
            incremental_results[
                incremental_results["Model"]
                ==
                model_name
            ]
            .copy()
        )

        # Force correct progression order
        model_inc["Configuration"] = (
            pd.Categorical(
                model_inc["Configuration"],
                categories=incremental_order,
                ordered=True
            )
        )

        model_inc = (
            model_inc
            .sort_values(
                "Configuration"
            )
            .reset_index(drop=True)
        )

        previous_auc = None

        for _, row in model_inc.iterrows():

            current_auc = (
                row["ROC_AUC_Mean"]
            )

            if previous_auc is None:
                gain = np.nan
            else:
                gain = (
                    current_auc
                    -
                    previous_auc
                )

            incremental_gain_rows.append({

                "Model":
                    model_name,

                "Configuration":
                    str(row["Configuration"]),

                "N_Features":
                    row["N_Features"],

                "ROC_AUC":
                    current_auc,

                "Gain_From_Previous_Step":
                    gain
            })

            previous_auc = current_auc

    incremental_gain_results = pd.DataFrame(
        incremental_gain_rows
    )

    print("=" * 85)
    print("INCREMENTAL ROC-AUC GAIN")
    print("=" * 85)

    display(
        incremental_gain_results
        .round(4)
    )

    print(
        "\n✅ Incremental ROC-AUC gains calculated."
    )

except Exception as e:
    print("\n❌ STEP 31 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 32: PLOT INCREMENTAL ROC-AUC
# ============================================================

try:
    incremental_order = list(
        INCREMENTAL_CONFIGS.keys()
    )

    plt.figure(
        figsize=(10, 6)
    )

    for model_name in ABLATION_MODELS.keys():

        temp = (
            incremental_results[
                incremental_results["Model"]
                ==
                model_name
            ]
            .copy()
        )

        temp["Configuration"] = pd.Categorical(
            temp["Configuration"],
            categories=incremental_order,
            ordered=True
        )

        temp = temp.sort_values(
            "Configuration"
        )

        plt.plot(
            range(len(temp)),
            temp["ROC_AUC_Mean"],
            marker="o",
            linewidth=2,
            label=model_name
        )

    plt.axhline(
        0.50,
        linestyle="--",
        linewidth=1.5,
        label="Chance = 0.50"
    )

    plt.xticks(
        range(
            len(incremental_order)
        ),
        incremental_order,
        rotation=25,
        ha="right"
    )

    plt.ylabel(
        "Mean 5-Fold ROC-AUC"
    )

    plt.xlabel(
        "Incremental Feature Configuration"
    )

    plt.title(
        "Incremental Contribution of Feature Groups"
    )

    plt.legend()

    plt.tight_layout()
    plt.show()

    print(
        "✅ Incremental contribution figure created."
    )

except Exception as e:
    print("\n❌ STEP 32 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))# ============================================================
# STEP 32: PLOT INCREMENTAL ROC-AUC
# ============================================================

try:
    incremental_order = list(
        INCREMENTAL_CONFIGS.keys()
    )

    plt.figure(
        figsize=(10, 6)
    )

    for model_name in ABLATION_MODELS.keys():

        temp = (
            incremental_results[
                incremental_results["Model"]
                ==
                model_name
            ]
            .copy()
        )

        temp["Configuration"] = pd.Categorical(
            temp["Configuration"],
            categories=incremental_order,
            ordered=True
        )

        temp = temp.sort_values(
            "Configuration"
        )

        plt.plot(
            range(len(temp)),
            temp["ROC_AUC_Mean"],
            marker="o",
            linewidth=2,
            label=model_name
        )

    plt.axhline(
        0.50,
        linestyle="--",
        linewidth=1.5,
        label="Chance = 0.50"
    )

    plt.xticks(
        range(
            len(incremental_order)
        ),
        incremental_order,
        rotation=25,
        ha="right"
    )

    plt.ylabel(
        "Mean 5-Fold ROC-AUC"
    )

    plt.xlabel(
        "Incremental Feature Configuration"
    )

    plt.title(
        "Incremental Contribution of Feature Groups"
    )

    plt.legend()

    plt.tight_layout()
    plt.show()

    print(
        "✅ Incremental contribution figure created."
    )

except Exception as e:
    print("\n❌ STEP 32 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 33: SAVE ABLATION EXPERIMENT RESULTS
# ============================================================

try:
    files_to_save = {

        "feature_ablation_summary.csv":
            ablation_results,

        "feature_ablation_fold_scores.csv":
            ablation_fold_results,

        "feature_ablation_deltas.csv":
            ablation_delta_results,

        "incremental_feature_summary.csv":
            incremental_results,

        "incremental_feature_fold_scores.csv":
            incremental_fold_results,

        "incremental_feature_gains.csv":
            incremental_gain_results
    }

    for filename, dataframe in (
        files_to_save.items()
    ):

        filepath = (
            f"{OUT_DIR}/"
            f"{filename}"
        )

        dataframe.to_csv(
            filepath,
            index=False
        )

        print(
            f"✅ Saved: {filename}"
        )

    print(
        "\n✅ All ablation results saved successfully."
    )

except Exception as e:
    print("\n❌ STEP 33 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 13. Statistical Comparison of Feature Configurations

The feature-group ablation experiment showed substantial differences in predictive discrimination depending on whether direct ocular symptoms were included.

To evaluate the robustness of these differences, a repeated cross-validation experiment is conducted using:

- 5 stratified folds
- 5 repetitions
- 25 paired ROC-AUC measurements per configuration

Two representative classifiers are evaluated:

- Logistic Regression
- XGBoost

Three planned comparisons are performed for each classifier:

1. All Features vs No Ocular Symptoms
2. Ocular Symptoms Only vs No Ocular Symptoms
3. All Features vs Ocular Symptoms Only

The paired Wilcoxon signed-rank test is used because the comparison is performed on matched resampling splits and does not require normally distributed score differences.

Because multiple comparisons are performed, Holm's method is used to adjust the p-values.

The held-out test set remains untouched during this analysis.

In [ ]:
# ============================================================
# STEP 34: REPEATED CROSS-VALIDATION SETUP
# ============================================================

try:
    from sklearn.model_selection import RepeatedStratifiedKFold

    # --------------------------------------------------------
    # 5 folds × 5 repetitions = 25 evaluations
    # --------------------------------------------------------
    repeated_cv = RepeatedStratifiedKFold(
        n_splits=5,
        n_repeats=5,
        random_state=RANDOM_STATE
    )

    print("=" * 75)
    print("REPEATED CROSS-VALIDATION SETUP")
    print("=" * 75)

    print("\nNumber of folds       : 5")
    print("Number of repetitions : 5")
    print("Total evaluations     : 25")
    print("Primary metric        : ROC-AUC")

    print("\n✅ Repeated CV configuration created successfully.")

except Exception as e:
    print("\n❌ STEP 34 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 13.1 Repeated Cross-Validation Scores

The same repeated stratified splits are used for every feature configuration.

Using identical resampling partitions makes the resulting ROC-AUC values paired, allowing direct comparison of feature configurations.

In [ ]:
# ============================================================
# STEP 35: REPEATED CV FOR STATISTICAL COMPARISON
# ============================================================

try:
    import time
    import numpy as np
    import pandas as pd

    from sklearn.model_selection import cross_val_score

    print("=" * 85)
    print("REPEATED CV — STATISTICAL COMPARISON")
    print("=" * 85)

    # --------------------------------------------------------
    # Only configurations needed for the main hypotheses
    # --------------------------------------------------------
    STAT_CONFIGS = {

        "All Features":
            ABLATION_CONFIGS["All Features"],

        "No Ocular Symptoms":
            ABLATION_CONFIGS["No Ocular Symptoms"],

        "Ocular Symptoms Only":
            ABLATION_CONFIGS["Ocular Symptoms Only"]
    }

    # --------------------------------------------------------
    # Representative models
    # --------------------------------------------------------
    STAT_MODELS = {

        "Logistic Regression":
            logistic_model,

        "XGBoost":
            xgboost_model
    }

    statistical_score_rows = []

    # --------------------------------------------------------
    # IMPORTANT:
    # Pre-generate the splits once.
    #
    # Every model/configuration will therefore be evaluated
    # on exactly the same train-validation partitions.
    # --------------------------------------------------------
    repeated_splits = list(
        repeated_cv.split(
            X_train,
            y_train
        )
    )

    print(
        f"\nGenerated paired splits: "
        f"{len(repeated_splits)}"
    )

    # --------------------------------------------------------
    # Run experiment
    # --------------------------------------------------------
    for model_name, model in STAT_MODELS.items():

        print("\n" + "-" * 85)
        print(model_name)
        print("-" * 85)

        for config_name, feature_list in STAT_CONFIGS.items():

            print(
                f"Running {config_name}...",
                end=" "
            )

            start_time = time.time()

            X_config = X_train[
                feature_list
            ].copy()

            scores = cross_val_score(
                estimator=model,
                X=X_config,
                y=y_train,

                cv=repeated_splits,

                scoring="roc_auc",

                # Models already make use of available CPUs
                n_jobs=1,

                error_score="raise"
            )

            elapsed = (
                time.time()
                -
                start_time
            )

            # Store every paired score
            for evaluation_number, score in enumerate(
                scores,
                start=1
            ):

                statistical_score_rows.append({

                    "Model":
                        model_name,

                    "Configuration":
                        config_name,

                    "Evaluation":
                        evaluation_number,

                    "ROC_AUC":
                        score
                })

            print(
                f"✅ Mean AUC = "
                f"{scores.mean():.4f} "
                f"± {scores.std():.4f} "
                f"({elapsed:.2f} sec)"
            )

    # --------------------------------------------------------
    # Build dataframe
    # --------------------------------------------------------
    repeated_cv_scores = pd.DataFrame(
        statistical_score_rows
    )

    # --------------------------------------------------------
    # Summary table
    # --------------------------------------------------------
    repeated_cv_summary = (
        repeated_cv_scores
        .groupby(
            ["Model", "Configuration"]
        )["ROC_AUC"]
        .agg([
            "mean",
            "std",
            "median",
            "min",
            "max"
        ])
        .reset_index()
    )

    repeated_cv_summary.columns = [
        "Model",
        "Configuration",
        "Mean_ROC_AUC",
        "STD",
        "Median",
        "Minimum",
        "Maximum"
    ]

    print("\n" + "=" * 85)
    print("REPEATED CV SUMMARY")
    print("=" * 85)

    display(
        repeated_cv_summary
        .round(4)
    )

    print(
        "\n✅ STEP 35 COMPLETED SUCCESSFULLY."
    )

except Exception as e:
    print("\n❌ STEP 35 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 13.2 Paired Statistical Tests

Three pre-specified comparisons are evaluated:

- **All Features vs No Ocular Symptoms**
- **Ocular Symptoms Only vs No Ocular Symptoms**
- **All Features vs Ocular Symptoms Only**

For each comparison, the ROC-AUC scores obtained from identical repeated cross-validation splits are paired.

The Wilcoxon signed-rank test evaluates whether the distribution of paired performance differences is centered around zero.

In [ ]:
# ============================================================
# STEP 36: PAIRED WILCOXON TESTS
# ============================================================

try:
    from scipy.stats import wilcoxon

    # --------------------------------------------------------
    # Planned comparisons
    # --------------------------------------------------------
    planned_comparisons = [

        (
            "All Features",
            "No Ocular Symptoms"
        ),

        (
            "Ocular Symptoms Only",
            "No Ocular Symptoms"
        ),

        (
            "All Features",
            "Ocular Symptoms Only"
        )
    ]

    statistical_test_rows = []

    print("=" * 90)
    print("PAIRED WILCOXON TESTS")
    print("=" * 90)

    for model_name in STAT_MODELS.keys():

        print(f"\nMODEL: {model_name}")
        print("-" * 90)

        model_scores = repeated_cv_scores[
            repeated_cv_scores["Model"]
            ==
            model_name
        ]

        for config_a, config_b in planned_comparisons:

            scores_a = (
                model_scores[
                    model_scores["Configuration"]
                    ==
                    config_a
                ]
                .sort_values("Evaluation")
                ["ROC_AUC"]
                .to_numpy()
            )

            scores_b = (
                model_scores[
                    model_scores["Configuration"]
                    ==
                    config_b
                ]
                .sort_values("Evaluation")
                ["ROC_AUC"]
                .to_numpy()
            )

            # -----------------------------------------------
            # Verify pairing
            # -----------------------------------------------
            if len(scores_a) != len(scores_b):
                raise ValueError(
                    "Paired score lengths do not match."
                )

            paired_difference = (
                scores_a
                -
                scores_b
            )

            # -----------------------------------------------
            # Wilcoxon signed-rank test
            # -----------------------------------------------
            test_result = wilcoxon(
                scores_a,
                scores_b,

                alternative="two-sided",

                zero_method="wilcox"
            )

            statistical_test_rows.append({

                "Model":
                    model_name,

                "Comparison":
                    f"{config_a} vs {config_b}",

                "Mean_AUC_A":
                    np.mean(scores_a),

                "Mean_AUC_B":
                    np.mean(scores_b),

                "Mean_Difference":
                    np.mean(
                        paired_difference
                    ),

                "Median_Difference":
                    np.median(
                        paired_difference
                    ),

                "Wilcoxon_Statistic":
                    test_result.statistic,

                "Raw_P_Value":
                    test_result.pvalue
            })

            print(
                f"{config_a} vs {config_b}"
            )

            print(
                f"  Mean ΔAUC = "
                f"{np.mean(paired_difference):.4f}"
            )

            print(
                f"  Raw p     = "
                f"{test_result.pvalue:.6f}"
            )

    statistical_tests = pd.DataFrame(
        statistical_test_rows
    )

    print(
        "\n✅ Raw paired statistical tests completed."
    )

except Exception as e:
    print("\n❌ STEP 36 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))# ============================================================
# STEP 36: PAIRED WILCOXON TESTS
# ============================================================

try:
    from scipy.stats import wilcoxon

    # --------------------------------------------------------
    # Planned comparisons
    # --------------------------------------------------------
    planned_comparisons = [

        (
            "All Features",
            "No Ocular Symptoms"
        ),

        (
            "Ocular Symptoms Only",
            "No Ocular Symptoms"
        ),

        (
            "All Features",
            "Ocular Symptoms Only"
        )
    ]

    statistical_test_rows = []

    print("=" * 90)
    print("PAIRED WILCOXON TESTS")
    print("=" * 90)

    for model_name in STAT_MODELS.keys():

        print(f"\nMODEL: {model_name}")
        print("-" * 90)

        model_scores = repeated_cv_scores[
            repeated_cv_scores["Model"]
            ==
            model_name
        ]

        for config_a, config_b in planned_comparisons:

            scores_a = (
                model_scores[
                    model_scores["Configuration"]
                    ==
                    config_a
                ]
                .sort_values("Evaluation")
                ["ROC_AUC"]
                .to_numpy()
            )

            scores_b = (
                model_scores[
                    model_scores["Configuration"]
                    ==
                    config_b
                ]
                .sort_values("Evaluation")
                ["ROC_AUC"]
                .to_numpy()
            )

            # -----------------------------------------------
            # Verify pairing
            # -----------------------------------------------
            if len(scores_a) != len(scores_b):
                raise ValueError(
                    "Paired score lengths do not match."
                )

            paired_difference = (
                scores_a
                -
                scores_b
            )

            # -----------------------------------------------
            # Wilcoxon signed-rank test
            # -----------------------------------------------
            test_result = wilcoxon(
                scores_a,
                scores_b,

                alternative="two-sided",

                zero_method="wilcox"
            )

            statistical_test_rows.append({

                "Model":
                    model_name,

                "Comparison":
                    f"{config_a} vs {config_b}",

                "Mean_AUC_A":
                    np.mean(scores_a),

                "Mean_AUC_B":
                    np.mean(scores_b),

                "Mean_Difference":
                    np.mean(
                        paired_difference
                    ),

                "Median_Difference":
                    np.median(
                        paired_difference
                    ),

                "Wilcoxon_Statistic":
                    test_result.statistic,

                "Raw_P_Value":
                    test_result.pvalue
            })

            print(
                f"{config_a} vs {config_b}"
            )

            print(
                f"  Mean ΔAUC = "
                f"{np.mean(paired_difference):.4f}"
            )

            print(
                f"  Raw p     = "
                f"{test_result.pvalue:.6f}"
            )

    statistical_tests = pd.DataFrame(
        statistical_test_rows
    )

    print(
        "\n✅ Raw paired statistical tests completed."
    )

except Exception as e:
    print("\n❌ STEP 36 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))# ============================================================
# STEP 36: PAIRED WILCOXON TESTS
# ============================================================

try:
    from scipy.stats import wilcoxon

    # --------------------------------------------------------
    # Planned comparisons
    # --------------------------------------------------------
    planned_comparisons = [

        (
            "All Features",
            "No Ocular Symptoms"
        ),

        (
            "Ocular Symptoms Only",
            "No Ocular Symptoms"
        ),

        (
            "All Features",
            "Ocular Symptoms Only"
        )
    ]

    statistical_test_rows = []

    print("=" * 90)
    print("PAIRED WILCOXON TESTS")
    print("=" * 90)

    for model_name in STAT_MODELS.keys():

        print(f"\nMODEL: {model_name}")
        print("-" * 90)

        model_scores = repeated_cv_scores[
            repeated_cv_scores["Model"]
            ==
            model_name
        ]

        for config_a, config_b in planned_comparisons:

            scores_a = (
                model_scores[
                    model_scores["Configuration"]
                    ==
                    config_a
                ]
                .sort_values("Evaluation")
                ["ROC_AUC"]
                .to_numpy()
            )

            scores_b = (
                model_scores[
                    model_scores["Configuration"]
                    ==
                    config_b
                ]
                .sort_values("Evaluation")
                ["ROC_AUC"]
                .to_numpy()
            )

            # -----------------------------------------------
            # Verify pairing
            # -----------------------------------------------
            if len(scores_a) != len(scores_b):
                raise ValueError(
                    "Paired score lengths do not match."
                )

            paired_difference = (
                scores_a
                -
                scores_b
            )

            # -----------------------------------------------
            # Wilcoxon signed-rank test
            # -----------------------------------------------
            test_result = wilcoxon(
                scores_a,
                scores_b,

                alternative="two-sided",

                zero_method="wilcox"
            )

            statistical_test_rows.append({

                "Model":
                    model_name,

                "Comparison":
                    f"{config_a} vs {config_b}",

                "Mean_AUC_A":
                    np.mean(scores_a),

                "Mean_AUC_B":
                    np.mean(scores_b),

                "Mean_Difference":
                    np.mean(
                        paired_difference
                    ),

                "Median_Difference":
                    np.median(
                        paired_difference
                    ),

                "Wilcoxon_Statistic":
                    test_result.statistic,

                "Raw_P_Value":
                    test_result.pvalue
            })

            print(
                f"{config_a} vs {config_b}"
            )

            print(
                f"  Mean ΔAUC = "
                f"{np.mean(paired_difference):.4f}"
            )

            print(
                f"  Raw p     = "
                f"{test_result.pvalue:.6f}"
            )

    statistical_tests = pd.DataFrame(
        statistical_test_rows
    )

    print(
        "\n✅ Raw paired statistical tests completed."
    )

except Exception as e:
    print("\n❌ STEP 36 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 13.3 Multiple-Comparison Correction

Three related comparisons are performed for each classifier.

To reduce the probability of false-positive statistical conclusions, Holm's step-down procedure is applied to the p-values separately for each classifier.

A corrected p-value below 0.05 is treated as evidence of a statistically detectable difference in the repeated resampling experiment.

In [ ]:
# ============================================================
# STEP 37: HOLM MULTIPLE-COMPARISON CORRECTION
# ============================================================

try:
    from statsmodels.stats.multitest import multipletests

    corrected_results = []

    for model_name in statistical_tests["Model"].unique():

        temp = (
            statistical_tests[
                statistical_tests["Model"]
                ==
                model_name
            ]
            .copy()
        )

        # -----------------------------------------------
        # Holm correction
        # -----------------------------------------------
        reject, corrected_p, _, _ = multipletests(

            temp["Raw_P_Value"],

            alpha=0.05,

            method="holm"
        )

        temp["Holm_Adjusted_P"] = (
            corrected_p
        )

        temp["Significant_0.05"] = (
            reject
        )

        corrected_results.append(
            temp
        )

    statistical_tests_corrected = pd.concat(
        corrected_results,
        ignore_index=True
    )

    print("=" * 95)
    print("STATISTICAL COMPARISON RESULTS")
    print("=" * 95)

    display(
        statistical_tests_corrected[
            [
                "Model",
                "Comparison",
                "Mean_AUC_A",
                "Mean_AUC_B",
                "Mean_Difference",
                "Median_Difference",
                "Raw_P_Value",
                "Holm_Adjusted_P",
                "Significant_0.05"
            ]
        ].round(6)
    )

    print(
        "\nNOTE:"
    )

    print(
        "These p-values provide resampling-based evidence "
        "within this dataset."
    )

    print(
        "Repeated CV folds are not completely independent, "
        "so the tests should support the effect-size analysis "
        "rather than be presented as population-level clinical inference."
    )

    print(
        "\n✅ STEP 37 COMPLETED SUCCESSFULLY."
    )

except Exception as e:
    print("\n❌ STEP 37 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 13.4 Effect Size and Uncertainty

Statistical significance alone does not indicate whether a difference is practically meaningful.

Therefore, the mean paired change in ROC-AUC is reported together with a bootstrap 95% interval across the repeated resampling results.

The magnitude and direction of the ROC-AUC difference remain the primary interpretation.

In [ ]:
# ============================================================
# STEP 38: BOOTSTRAP INTERVALS FOR PAIRED AUC DIFFERENCES
# ============================================================

try:
    rng = np.random.default_rng(
        RANDOM_STATE
    )

    N_BOOTSTRAPS = 5000

    bootstrap_rows = []

    for model_name in STAT_MODELS.keys():

        model_scores = repeated_cv_scores[
            repeated_cv_scores["Model"]
            ==
            model_name
        ]

        for config_a, config_b in planned_comparisons:

            scores_a = (
                model_scores[
                    model_scores["Configuration"]
                    ==
                    config_a
                ]
                .sort_values("Evaluation")
                ["ROC_AUC"]
                .to_numpy()
            )

            scores_b = (
                model_scores[
                    model_scores["Configuration"]
                    ==
                    config_b
                ]
                .sort_values("Evaluation")
                ["ROC_AUC"]
                .to_numpy()
            )

            differences = (
                scores_a
                -
                scores_b
            )

            # -----------------------------------------------
            # Bootstrap paired differences
            # -----------------------------------------------
            bootstrap_means = []

            n = len(differences)

            for _ in range(N_BOOTSTRAPS):

                sampled_indices = rng.integers(
                    0,
                    n,
                    size=n
                )

                bootstrap_means.append(
                    differences[
                        sampled_indices
                    ].mean()
                )

            bootstrap_means = np.array(
                bootstrap_means
            )

            ci_lower = np.percentile(
                bootstrap_means,
                2.5
            )

            ci_upper = np.percentile(
                bootstrap_means,
                97.5
            )

            bootstrap_rows.append({

                "Model":
                    model_name,

                "Comparison":
                    f"{config_a} vs {config_b}",

                "Mean_Delta_ROC_AUC":
                    differences.mean(),

                "CI_95_Lower":
                    ci_lower,

                "CI_95_Upper":
                    ci_upper
            })

    auc_difference_intervals = pd.DataFrame(
        bootstrap_rows
    )

    print("=" * 95)
    print("PAIRED AUC DIFFERENCE — EFFECT SIZE")
    print("=" * 95)

    display(
        auc_difference_intervals
        .round(5)
    )

    print(
        "\n✅ Effect-size intervals calculated successfully."
    )

except Exception as e:
    print("\n❌ STEP 38 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 39: MAIN STATISTICAL FINDING SUMMARY
# ============================================================

try:
    print("=" * 90)
    print("MAIN FEATURE-GROUP FINDINGS")
    print("=" * 90)

    for model_name in STAT_MODELS.keys():

        print(f"\n{model_name}")
        print("-" * 90)

        model_results = statistical_tests_corrected[
            statistical_tests_corrected["Model"]
            ==
            model_name
        ]

        for _, row in model_results.iterrows():

            significance_text = (
                "statistically detectable"
                if row["Significant_0.05"]
                else
                "not statistically detectable"
            )

            print(
                f"\n{row['Comparison']}"
            )

            print(
                f"Mean ΔROC-AUC       : "
                f"{row['Mean_Difference']:.4f}"
            )

            print(
                f"Holm-adjusted p     : "
                f"{row['Holm_Adjusted_P']:.6f}"
            )

            print(
                f"Result              : "
                f"{significance_text}"
            )

    print(
        "\nIMPORTANT:"
    )

    print(
        "The practical size of the ROC-AUC change should "
        "be emphasized more strongly than the p-value."
    )

    print(
        "\n✅ Statistical interpretation summary completed."
    )

except Exception as e:
    print("\n❌ STEP 39 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 40: SAVE STATISTICAL COMPARISON RESULTS
# ============================================================

try:
    files_to_save = {

        "repeated_cv_auc_scores.csv":
            repeated_cv_scores,

        "repeated_cv_auc_summary.csv":
            repeated_cv_summary,

        "statistical_comparison_tests.csv":
            statistical_tests_corrected,

        "auc_difference_intervals.csv":
            auc_difference_intervals
    }

    for filename, dataframe in files_to_save.items():

        filepath = (
            f"{OUT_DIR}/"
            f"{filename}"
        )

        dataframe.to_csv(
            filepath,
            index=False
        )

        print(
            f"✅ Saved: {filename}"
        )

    print(
        "\n✅ Statistical comparison outputs saved."
    )

except Exception as e:
    print("\n❌ STEP 40 SAVE FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 14. SHAP-Based Model Explainability

SHAP (SHapley Additive exPlanations) is used to interpret the full-feature XGBoost model.

The analysis addresses two questions:

1. Which individual predictors contribute most strongly to the model's decisions?
2. Which predefined feature groups contribute most strongly overall?

The SHAP analysis is performed using the training data only.

The held-out test set remains untouched.

In [ ]:
# ============================================================
# STEP 41: CHECK SHAP LIBRARY
# ============================================================

try:
    import shap

    print("=" * 75)
    print("SHAP LIBRARY CHECK")
    print("=" * 75)

    print(f"\nSHAP version: {shap.__version__}")
    print("✅ SHAP is available.")

except ImportError as e:

    print("\n❌ SHAP is not installed.")
    print("Error:", str(e))

    print(
        "\nRun this in a NEW code cell:"
    )

    print("pip install -q shap")

    print(
        "\nThen rerun this cell."
    )

### 14.1 Full-Feature XGBoost Explanation Model

A fresh XGBoost model is trained using all 26 predictors from the training set.

This model is used only for explainability analysis and does not use the held-out test set.

In [ ]:
# ============================================================
# STEP 42: FIT FULL-FEATURE XGBOOST FOR SHAP
# ============================================================

try:
    from sklearn.base import clone

    print("=" * 80)
    print("TRAINING FULL-FEATURE XGBOOST FOR SHAP")
    print("=" * 80)

    # --------------------------------------------------------
    # Use all 26 pre-defined predictors
    # --------------------------------------------------------
    X_train_all = X_train[
        ABLATION_CONFIGS["All Features"]
    ].copy()

    # --------------------------------------------------------
    # Clone the existing model so we do not overwrite
    # previous notebook objects
    # --------------------------------------------------------
    xgb_shap_model = clone(
        xgboost_model
    )

    # --------------------------------------------------------
    # Fit using TRAINING DATA ONLY
    # --------------------------------------------------------
    xgb_shap_model.fit(
        X_train_all,
        y_train
    )

    print(
        f"\nTraining observations : "
        f"{X_train_all.shape[0]:,}"
    )

    print(
        f"Predictors            : "
        f"{X_train_all.shape[1]}"
    )

    print("\n✅ XGBoost SHAP model fitted successfully.")
    print("✅ Held-out test data has NOT been used.")

except Exception as e:
    print("\n❌ STEP 42 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 14.2 SHAP Analysis Sample

To make SHAP computation efficient while retaining a representative sample, 3,000 observations are randomly selected from the training dataset.

The sampling process uses a fixed random seed for reproducibility.

In [ ]:
# ============================================================
# STEP 43: CREATE SHAP ANALYSIS SAMPLE
# ============================================================

try:
    SHAP_SAMPLE_SIZE = min(
        3000,
        len(X_train_all)
    )

    X_shap = X_train_all.sample(
        n=SHAP_SAMPLE_SIZE,
        random_state=RANDOM_STATE
    ).copy()

    print("=" * 75)
    print("SHAP SAMPLE")
    print("=" * 75)

    print(
        f"\nSHAP sample size : "
        f"{len(X_shap):,}"
    )

    print(
        f"Number of features: "
        f"{X_shap.shape[1]}"
    )

    print("\n✅ SHAP sample created successfully.")

except Exception as e:
    print("\n❌ STEP 43 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 44: COMPUTE SHAP VALUES
# CHECKPOINT 6A
# ============================================================

try:
    import shap
    import numpy as np

    print("=" * 80)
    print("COMPUTING SHAP VALUES")
    print("=" * 80)

    # --------------------------------------------------------
    # TreeExplainer is optimized for tree-based models
    # such as XGBoost.
    # --------------------------------------------------------
    explainer = shap.TreeExplainer(
        xgb_shap_model
    )

    shap_values = explainer(
        X_shap
    )

    # --------------------------------------------------------
    # Basic integrity checks
    # --------------------------------------------------------
    print(
        f"\nSHAP values shape : "
        f"{shap_values.values.shape}"
    )

    print(
        f"Feature matrix shape: "
        f"{X_shap.shape}"
    )

    # Most binary XGBoost models return:
    # (n_samples, n_features)
    #
    # If a future SHAP/XGBoost version returns an extra
    # output dimension, stop rather than silently reshaping.
    if shap_values.values.ndim != 2:
        raise ValueError(
            "Unexpected SHAP output dimensionality: "
            f"{shap_values.values.shape}"
        )

    if (
        shap_values.values.shape[1]
        !=
        X_shap.shape[1]
    ):
        raise ValueError(
            "Number of SHAP feature columns does not "
            "match the input feature matrix."
        )

    print("\n✅ SHAP values computed successfully.")
    print("✅ CHECKPOINT 6A PASSED.")

except Exception as e:
    print("\n❌ CHECKPOINT 6A FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

    print(
        "\nPlease send me the exact error before "
        "changing the code."
    )

### 14.3 Global Feature Importance

Global SHAP importance is calculated as the mean absolute SHAP value for each predictor.

A larger mean absolute SHAP value indicates that a predictor has a greater average influence on the XGBoost model's output.

This measures model influence, not causality.

In [ ]:
# ============================================================
# STEP 45: GLOBAL SHAP FEATURE RANKING
# ============================================================

try:
    # --------------------------------------------------------
    # Mean absolute SHAP value for each feature
    # --------------------------------------------------------
    mean_abs_shap = np.abs(
        shap_values.values
    ).mean(axis=0)

    shap_feature_importance = pd.DataFrame({

        "Feature":
            X_shap.columns,

        "Mean_Absolute_SHAP":
            mean_abs_shap

    }).sort_values(
        "Mean_Absolute_SHAP",
        ascending=False
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Add rank
    # --------------------------------------------------------
    shap_feature_importance.insert(
        0,
        "Rank",
        range(
            1,
            len(shap_feature_importance) + 1
        )
    )

    # --------------------------------------------------------
    # Add relative percentage contribution
    # --------------------------------------------------------
    total_shap = (
        shap_feature_importance[
            "Mean_Absolute_SHAP"
        ].sum()
    )

    shap_feature_importance[
        "Relative_Importance_Percent"
    ] = (
        shap_feature_importance[
            "Mean_Absolute_SHAP"
        ]
        /
        total_shap
        *
        100
    )

    print("=" * 85)
    print("GLOBAL SHAP FEATURE IMPORTANCE")
    print("=" * 85)

    display(
        shap_feature_importance
        .head(15)
        .round(5)
    )

    print(
        "\n✅ Global SHAP ranking created successfully."
    )

except Exception as e:
    print("\n❌ STEP 45 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 46: SHAP GLOBAL BAR PLOT
# ============================================================

try:
    shap.plots.bar(
        shap_values,
        max_display=15,
        show=False
    )

    plt.title(
        "Global SHAP Feature Importance — XGBoost"
    )

    plt.tight_layout()
    plt.show()

    print(
        "✅ SHAP global importance plot created."
    )

except Exception as e:
    print("\n❌ STEP 46 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 14.4 SHAP Beeswarm Plot

The SHAP beeswarm plot shows both feature importance and direction of influence.

Each point represents one observation:

- positive SHAP values push the model toward the Dry Eye class;
- negative SHAP values push the model toward the non-Dry-Eye class;
- feature values are represented along the color scale.

This plot helps determine whether the model's predictions are primarily driven by direct ocular symptoms.

In [ ]:
# ============================================================
# STEP 47: SHAP BEESWARM PLOT
# ============================================================

try:
    shap.plots.beeswarm(
        shap_values,
        max_display=15,
        show=False
    )

    plt.title(
        "SHAP Summary of XGBoost Dry Eye Predictions"
    )

    plt.tight_layout()
    plt.show()

    print(
        "✅ SHAP beeswarm plot created successfully."
    )

except Exception as e:
    print("\n❌ STEP 47 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 14.5 SHAP Contribution by Feature Group

Individual SHAP values are aggregated according to the five predefined feature groups:

- Demographic
- Sleep / Health
- Lifestyle
- Digital Exposure
- Ocular Symptoms

The purpose is to quantify how much of the model's total average feature influence is associated with each conceptual predictor group.### 14.5 SHAP Contribution by Feature Group

Individual SHAP values are aggregated according to the five predefined feature groups:

- Demographic
- Sleep / Health
- Lifestyle
- Digital Exposure
- Ocular Symptoms

The purpose is to quantify how much of the model's total average feature influence is associated with each conceptual predictor group.

In [ ]:
# ============================================================
# STEP 48: AGGREGATE SHAP IMPORTANCE BY FEATURE GROUP
# CHECKPOINT 6B
# ============================================================

try:
    # --------------------------------------------------------
    # Map every feature to its predefined feature group
    # --------------------------------------------------------
    feature_to_group = {}

    for group_name, features in FEATURE_GROUPS.items():

        for feature in features:

            feature_to_group[
                feature
            ] = group_name

    # --------------------------------------------------------
    # Add group label to SHAP table
    # --------------------------------------------------------
    shap_feature_importance[
        "Feature_Group"
    ] = (
        shap_feature_importance[
            "Feature"
        ]
        .map(feature_to_group)
    )

    # Safety check
    if (
        shap_feature_importance[
            "Feature_Group"
        ]
        .isna()
        .any()
    ):

        missing_group_features = (
            shap_feature_importance.loc[
                shap_feature_importance[
                    "Feature_Group"
                ].isna(),
                "Feature"
            ]
            .tolist()
        )

        raise ValueError(
            "Some features could not be mapped "
            f"to a group: {missing_group_features}"
        )

    # --------------------------------------------------------
    # Aggregate mean absolute SHAP values
    # --------------------------------------------------------
    shap_group_importance = (
        shap_feature_importance
        .groupby(
            "Feature_Group",
            as_index=False
        )["Mean_Absolute_SHAP"]
        .sum()
    )

    total_group_shap = (
        shap_group_importance[
            "Mean_Absolute_SHAP"
        ]
        .sum()
    )

    shap_group_importance[
        "Relative_Contribution_Percent"
    ] = (
        shap_group_importance[
            "Mean_Absolute_SHAP"
        ]
        /
        total_group_shap
        *
        100
    )

    shap_group_importance = (
        shap_group_importance
        .sort_values(
            "Relative_Contribution_Percent",
            ascending=False
        )
        .reset_index(drop=True)
    )

    shap_group_importance.insert(
        0,
        "Rank",
        range(
            1,
            len(shap_group_importance) + 1
        )
    )

    print("=" * 85)
    print("SHAP IMPORTANCE BY FEATURE GROUP")
    print("=" * 85)

    display(
        shap_group_importance
        .round(4)
    )

    print(
        "\n✅ Feature-group SHAP analysis completed."
    )

    print(
        "✅ CHECKPOINT 6B PASSED."
    )

except Exception as e:
    print("\n❌ CHECKPOINT 6B FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 49: PLOT FEATURE-GROUP SHAP CONTRIBUTION
# ============================================================

try:
    plt.figure(
        figsize=(9, 5)
    )

    bars = plt.bar(
        shap_group_importance[
            "Feature_Group"
        ],

        shap_group_importance[
            "Relative_Contribution_Percent"
        ]
    )

    # --------------------------------------------------------
    # Percentage labels
    # --------------------------------------------------------
    for bar, value in zip(
        bars,
        shap_group_importance[
            "Relative_Contribution_Percent"
        ]
    ):

        plt.text(
            bar.get_x()
            +
            bar.get_width() / 2,

            bar.get_height(),

            f"{value:.1f}%",

            ha="center",
            va="bottom"
        )

    plt.ylabel(
        "Share of Total Mean Absolute SHAP (%)"
    )

    plt.xlabel(
        "Feature Group"
    )

    plt.title(
        "XGBoost SHAP Contribution by Feature Group"
    )

    plt.xticks(
        rotation=20,
        ha="right"
    )

    plt.tight_layout()
    plt.show()

    print(
        "✅ SHAP feature-group plot created."
    )

except Exception as e:
    print("\n❌ STEP 49 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 14.6 Comparison with Univariate Associations

To determine whether the model-based SHAP ranking is consistent with the earlier exploratory analysis, the strongest SHAP features are compared with their univariate correlations with Dry Eye Disease.

Agreement between the two rankings would provide converging evidence that the same variables dominate both simple association and model-based prediction.

In [ ]:
# ============================================================
# STEP 50: COMPARE SHAP WITH TARGET ASSOCIATIONS
# ============================================================

try:
    shap_association_comparison = (
        shap_feature_importance[
            [
                "Rank",
                "Feature",
                "Mean_Absolute_SHAP",
                "Relative_Importance_Percent",
                "Feature_Group"
            ]
        ]
        .merge(
            association_table[
                [
                    "Correlation",
                    "Absolute_Correlation"
                ]
            ],
            left_on="Feature",
            right_index=True,
            how="left"
        )
    )

    print("=" * 90)
    print("SHAP VS UNIVARIATE ASSOCIATION")
    print("=" * 90)

    display(
        shap_association_comparison
        .head(15)
        .round(5)
    )

    print(
        "\n✅ SHAP and association rankings combined."
    )

except Exception as e:
    print("\n❌ STEP 50 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 14.7 Example Individual Prediction

A single training observation with a relatively high predicted probability of Dry Eye Disease is selected to demonstrate a local SHAP explanation.

This example illustrates how individual predictors combine to influence one model prediction.

The example is descriptive and should not be interpreted as a clinical diagnosis.

In [ ]:
# ============================================================
# STEP 51: LOCAL SHAP EXPLANATION
# ============================================================

try:
    # --------------------------------------------------------
    # Predict probabilities for SHAP sample
    # --------------------------------------------------------
    shap_probabilities = (
        xgb_shap_model
        .predict_proba(
            X_shap
        )[:, 1]
    )

    # --------------------------------------------------------
    # Select observation with highest predicted probability
    # --------------------------------------------------------
    highest_probability_position = (
        np.argmax(
            shap_probabilities
        )
    )

    selected_probability = (
        shap_probabilities[
            highest_probability_position
        ]
    )

    print("=" * 80)
    print("LOCAL SHAP EXAMPLE")
    print("=" * 80)

    print(
        f"\nPredicted Dry Eye probability: "
        f"{selected_probability:.4f}"
    )

    # --------------------------------------------------------
    # Waterfall plot
    # --------------------------------------------------------
    shap.plots.waterfall(
        shap_values[
            highest_probability_position
        ],

        max_display=12,

        show=False
    )

    plt.tight_layout()
    plt.show()

    print(
        "✅ Local SHAP explanation created."
    )

except Exception as e:
    print("\n❌ STEP 51 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 52: SAVE SHAP RESULTS
# ============================================================

try:
    shap_feature_importance.to_csv(
        f"{OUT_DIR}/"
        "shap_feature_importance.csv",

        index=False
    )

    shap_group_importance.to_csv(
        f"{OUT_DIR}/"
        "shap_group_importance.csv",

        index=False
    )

    shap_association_comparison.to_csv(
        f"{OUT_DIR}/"
        "shap_association_comparison.csv",

        index=False
    )

    print("✅ SHAP result tables saved.")

    print("\nFiles:")
    print("- shap_feature_importance.csv")
    print("- shap_group_importance.csv")
    print("- shap_association_comparison.csv")

except Exception as e:
    print("\n❌ STEP 52 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 15. Feature-Importance Stability Analysis

A single SHAP analysis can be sensitive to the particular sample used to fit a model.

Therefore, the stability of feature importance is evaluated using repeated stratified cross-validation.

The procedure uses:

- 5 stratified folds
- 3 repetitions
- 15 independently fitted XGBoost models

For each split:

1. XGBoost is fitted on the training portion.
2. SHAP values are calculated on the corresponding validation portion.
3. Mean absolute SHAP importance is calculated for every predictor.
4. Predictors are ranked within the split.

Feature stability is then summarized using:

- mean SHAP importance;
- standard deviation of SHAP importance;
- median feature rank;
- rank variability;
- frequency of appearing in the Top 3;
- frequency of appearing in the Top 5;
- frequency of appearing in the Top 10.

The held-out final test set is not used.

In [ ]:
# ============================================================
# STEP 53: FEATURE IMPORTANCE STABILITY SETUP
# ============================================================

try:
    from sklearn.model_selection import RepeatedStratifiedKFold
    from sklearn.base import clone

    stability_cv = RepeatedStratifiedKFold(
        n_splits=5,
        n_repeats=3,
        random_state=RANDOM_STATE
    )

    stability_splits = list(
        stability_cv.split(
            X_train_all,
            y_train
        )
    )

    print("=" * 80)
    print("FEATURE IMPORTANCE STABILITY SETUP")
    print("=" * 80)

    print(
        f"\nNumber of folds       : 5"
    )

    print(
        f"Number of repetitions : 3"
    )

    print(
        f"Total fitted models   : "
        f"{len(stability_splits)}"
    )

    print(
        f"Number of predictors  : "
        f"{X_train_all.shape[1]}"
    )

    if len(stability_splits) != 15:
        raise ValueError(
            "Expected exactly 15 resampling splits."
        )

    print(
        "\n✅ Stability analysis setup completed."
    )

except Exception as e:
    print("\n❌ STEP 53 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 15.1 Repeated Validation-Fold SHAP Importance

For each repeated cross-validation split, the model is trained without access to that split's validation observations.

SHAP importance is then computed on the validation fold.

This produces 15 independent feature-importance rankings and allows the consistency of the model's explanations to be evaluated.

In [ ]:
# ============================================================
# STEP 54: REPEATED SHAP FEATURE IMPORTANCE
# CHECKPOINT 7A
# ============================================================

try:
    import time
    import shap

    stability_rows = []

    print("=" * 90)
    print("REPEATED SHAP FEATURE IMPORTANCE")
    print("=" * 90)

    for split_number, (
        train_idx,
        valid_idx
    ) in enumerate(
        stability_splits,
        start=1
    ):

        start_time = time.time()

        # ----------------------------------------------------
        # Training and validation partitions
        # ----------------------------------------------------
        X_fold_train = (
            X_train_all
            .iloc[train_idx]
            .copy()
        )

        y_fold_train = (
            y_train
            .iloc[train_idx]
            .copy()
        )

        X_fold_valid = (
            X_train_all
            .iloc[valid_idx]
            .copy()
        )

        # ----------------------------------------------------
        # Fit a fresh XGBoost model
        # ----------------------------------------------------
        fold_model = clone(
            xgboost_model
        )

        fold_model.fit(
            X_fold_train,
            y_fold_train
        )

        # ----------------------------------------------------
        # Explain only validation observations
        # ----------------------------------------------------
        fold_explainer = shap.TreeExplainer(
            fold_model
        )

        fold_shap = fold_explainer(
            X_fold_valid
        )

        if fold_shap.values.ndim != 2:
            raise ValueError(
                "Unexpected SHAP dimensionality "
                f"in split {split_number}: "
                f"{fold_shap.values.shape}"
            )

        # ----------------------------------------------------
        # Mean absolute SHAP per feature
        # ----------------------------------------------------
        fold_importance = np.abs(
            fold_shap.values
        ).mean(axis=0)

        fold_table = pd.DataFrame({

            "Feature":
                X_fold_valid.columns,

            "Mean_Absolute_SHAP":
                fold_importance
        })

        # Largest importance receives rank 1
        fold_table["Rank"] = (
            fold_table[
                "Mean_Absolute_SHAP"
            ]
            .rank(
                ascending=False,
                method="min"
            )
            .astype(int)
        )

        # ----------------------------------------------------
        # Record repetition and fold numbers
        # ----------------------------------------------------
        repetition_number = (
            (split_number - 1) // 5
            + 1
        )

        fold_number = (
            (split_number - 1) % 5
            + 1
        )

        fold_table[
            "Split"
        ] = split_number

        fold_table[
            "Repetition"
        ] = repetition_number

        fold_table[
            "Fold"
        ] = fold_number

        stability_rows.append(
            fold_table
        )

        elapsed = (
            time.time()
            -
            start_time
        )

        # ----------------------------------------------------
        # Show top three features from this split
        # ----------------------------------------------------
        top_three = (
            fold_table
            .sort_values("Rank")
            .head(3)
            ["Feature"]
            .tolist()
        )

        print(
            f"Split {split_number:02d}/15 "
            f"| Top 3: {top_three} "
            f"| {elapsed:.2f} sec"
        )

    # --------------------------------------------------------
    # Combine all 15 importance tables
    # --------------------------------------------------------
    shap_stability_fold_importance = (
        pd.concat(
            stability_rows,
            ignore_index=True
        )
    )

    expected_rows = (
        15
        *
        X_train_all.shape[1]
    )

    if (
        len(shap_stability_fold_importance)
        !=
        expected_rows
    ):
        raise ValueError(
            f"Expected {expected_rows} rows but found "
            f"{len(shap_stability_fold_importance)}."
        )

    print("\n" + "=" * 90)

    print(
        f"Total stability records: "
        f"{len(shap_stability_fold_importance)}"
    )

    print(
        "✅ Repeated SHAP computation completed."
    )

    print(
        "✅ CHECKPOINT 7A PASSED."
    )

except Exception as e:
    print("\n❌ CHECKPOINT 7A FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

    print(
        "\nSend me the exact error before modifying "
        "the methodology."
    )

### 15.2 Feature Ranking Stability

The repeated SHAP results are summarized across all 15 fitted models.

A highly stable predictor should show:

- a low median rank;
- low rank variability;
- frequent appearance among the highest-ranked predictors.

This analysis distinguishes consistently important variables from variables that appear important only under particular resampling partitions.

In [ ]:
# ============================================================
# STEP 55: SUMMARIZE FEATURE IMPORTANCE STABILITY
# ============================================================

try:
    stability_summary_rows = []

    for feature in X_train_all.columns:

        temp = (
            shap_stability_fold_importance[
                shap_stability_fold_importance[
                    "Feature"
                ]
                ==
                feature
            ]
        )

        ranks = (
            temp["Rank"]
            .to_numpy()
        )

        shap_scores = (
            temp[
                "Mean_Absolute_SHAP"
            ]
            .to_numpy()
        )

        stability_summary_rows.append({

            "Feature":
                feature,

            "Mean_SHAP":
                shap_scores.mean(),

            "SHAP_STD":
                shap_scores.std(),

            "Median_Rank":
                np.median(ranks),

            "Mean_Rank":
                ranks.mean(),

            "Rank_STD":
                ranks.std(),

            "Best_Rank":
                ranks.min(),

            "Worst_Rank":
                ranks.max(),

            "Top_3_Frequency_Percent":
                np.mean(
                    ranks <= 3
                )
                *
                100,

            "Top_5_Frequency_Percent":
                np.mean(
                    ranks <= 5
                )
                *
                100,

            "Top_10_Frequency_Percent":
                np.mean(
                    ranks <= 10
                )
                *
                100
        })

    shap_stability_summary = pd.DataFrame(
        stability_summary_rows
    )

    # --------------------------------------------------------
    # Add semantic feature group
    # --------------------------------------------------------
    shap_stability_summary[
        "Feature_Group"
    ] = (
        shap_stability_summary[
            "Feature"
        ]
        .map(feature_to_group)
    )

    # --------------------------------------------------------
    # Sort primarily by median rank and then mean rank
    # --------------------------------------------------------
    shap_stability_summary = (
        shap_stability_summary
        .sort_values(
            [
                "Median_Rank",
                "Mean_Rank"
            ]
        )
        .reset_index(drop=True)
    )

    print("=" * 100)
    print("FEATURE IMPORTANCE STABILITY SUMMARY")
    print("=" * 100)

    display(
        shap_stability_summary
        .head(15)
        .round(4)
    )

    print(
        "\n✅ Feature stability summary created."
    )

except Exception as e:
    print("\n❌ STEP 55 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 56: OCULAR FEATURE STABILITY CHECK
# CHECKPOINT 7B
# ============================================================

try:
    ocular_stability = (
        shap_stability_summary[
            shap_stability_summary[
                "Feature"
            ]
            .isin(
                FEATURE_GROUPS[
                    "Ocular_Symptoms"
                ]
            )
        ]
        .copy()
        .sort_values(
            "Median_Rank"
        )
    )

    print("=" * 95)
    print("OCULAR SYMPTOM STABILITY")
    print("=" * 95)

    display(
        ocular_stability[
            [
                "Feature",
                "Mean_SHAP",
                "SHAP_STD",
                "Median_Rank",
                "Mean_Rank",
                "Best_Rank",
                "Worst_Rank",
                "Top_3_Frequency_Percent",
                "Top_5_Frequency_Percent"
            ]
        ].round(4)
    )

    # --------------------------------------------------------
    # Count splits in which ALL three ocular symptoms
    # simultaneously occupy the top three positions.
    # --------------------------------------------------------
    expected_ocular = set(
        FEATURE_GROUPS[
            "Ocular_Symptoms"
        ]
    )

    all_ocular_top3_count = 0

    for split_number in range(
        1,
        16
    ):

        split_top3 = set(

            shap_stability_fold_importance[
                shap_stability_fold_importance[
                    "Split"
                ]
                ==
                split_number
            ]
            .sort_values("Rank")
            .head(3)
            ["Feature"]
        )

        if (
            split_top3
            ==
            expected_ocular
        ):
            all_ocular_top3_count += 1

    all_ocular_top3_percent = (
        all_ocular_top3_count
        /
        15
        *
        100
    )

    print(
        "\nSplits where ALL three ocular symptoms "
        "occupied the Top 3:"
    )

    print(
        f"{all_ocular_top3_count}/15 "
        f"({all_ocular_top3_percent:.1f}%)"
    )

    print(
        "\n✅ CHECKPOINT 7B PASSED."
    )

except Exception as e:
    print("\n❌ CHECKPOINT 7B FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 15.3 Stability of Feature-Group Importance

SHAP importance is also aggregated within the predefined conceptual groups for each resampling split.

Both total group importance and average importance per feature are calculated.

The average-per-feature measure is included because the feature groups contain different numbers of predictors.

In [ ]:
# ============================================================
# STEP 57: FEATURE-GROUP STABILITY
# ============================================================

try:
    group_stability_rows = []

    for split_number in sorted(
        shap_stability_fold_importance[
            "Split"
        ].unique()
    ):

        split_data = (
            shap_stability_fold_importance[
                shap_stability_fold_importance[
                    "Split"
                ]
                ==
                split_number
            ]
            .copy()
        )

        split_data[
            "Feature_Group"
        ] = (
            split_data[
                "Feature"
            ]
            .map(
                feature_to_group
            )
        )

        group_table = (
            split_data
            .groupby(
                "Feature_Group"
            )
            .agg(

                Total_Mean_Absolute_SHAP=(
                    "Mean_Absolute_SHAP",
                    "sum"
                ),

                Average_Per_Feature_SHAP=(
                    "Mean_Absolute_SHAP",
                    "mean"
                ),

                Number_of_Features=(
                    "Feature",
                    "count"
                )
            )
            .reset_index()
        )

        total_shap = (
            group_table[
                "Total_Mean_Absolute_SHAP"
            ]
            .sum()
        )

        group_table[
            "Contribution_Percent"
        ] = (
            group_table[
                "Total_Mean_Absolute_SHAP"
            ]
            /
            total_shap
            *
            100
        )

        group_table[
            "Group_Rank"
        ] = (
            group_table[
                "Total_Mean_Absolute_SHAP"
            ]
            .rank(
                ascending=False,
                method="min"
            )
            .astype(int)
        )

        group_table[
            "Split"
        ] = split_number

        group_stability_rows.append(
            group_table
        )

    shap_group_stability_folds = (
        pd.concat(
            group_stability_rows,
            ignore_index=True
        )
    )

    # --------------------------------------------------------
    # Summarize across 15 splits
    # --------------------------------------------------------
    shap_group_stability_summary = (
        shap_group_stability_folds
        .groupby(
            "Feature_Group"
        )
        .agg(

            Mean_Contribution_Percent=(
                "Contribution_Percent",
                "mean"
            ),

            Contribution_STD=(
                "Contribution_Percent",
                "std"
            ),

            Mean_Per_Feature_SHAP=(
                "Average_Per_Feature_SHAP",
                "mean"
            ),

            Median_Group_Rank=(
                "Group_Rank",
                "median"
            ),

            Mean_Group_Rank=(
                "Group_Rank",
                "mean"
            ),

            Rank_1_Frequency_Percent=(
                "Group_Rank",
                lambda x:
                    np.mean(
                        np.asarray(x)
                        ==
                        1
                    )
                    *
                    100
            )
        )
        .reset_index()
        .sort_values(
            "Mean_Group_Rank"
        )
    )

    print("=" * 95)
    print("FEATURE-GROUP STABILITY")
    print("=" * 95)

    display(
        shap_group_stability_summary
        .round(4)
    )

    print(
        "\n✅ Feature-group stability analysis completed."
    )

except Exception as e:
    print("\n❌ STEP 57 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 15.4 Distribution of SHAP Importance Across Resampling Splits

The distribution of mean absolute SHAP values across repeated validation folds is visualized for the highest-ranked predictors.

A narrow distribution indicates that the importance estimate is relatively stable across different samples.

In [ ]:
# ============================================================
# STEP 58: VISUALIZE FEATURE IMPORTANCE STABILITY
# ============================================================

try:
    # --------------------------------------------------------
    # Select top 10 features according to the stability summary
    # --------------------------------------------------------
    top_10_stable_features = (
        shap_stability_summary
        .head(10)
        ["Feature"]
        .tolist()
    )

    plot_stability = (
        shap_stability_fold_importance[
            shap_stability_fold_importance[
                "Feature"
            ]
            .isin(
                top_10_stable_features
            )
        ]
        .copy()
    )

    # Fixed order based on stability ranking
    plot_stability[
        "Feature"
    ] = pd.Categorical(

        plot_stability[
            "Feature"
        ],

        categories=top_10_stable_features,

        ordered=True
    )

    plt.figure(
        figsize=(11, 6)
    )

    sns.boxplot(
        data=plot_stability,
        x="Feature",
        y="Mean_Absolute_SHAP"
    )

    plt.title(
        "Stability of SHAP Importance Across 15 Resampling Splits"
    )

    plt.xlabel("")

    plt.ylabel(
        "Mean Absolute SHAP"
    )

    plt.xticks(
        rotation=35,
        ha="right"
    )

    plt.tight_layout()
    plt.show()

    print(
        "✅ SHAP stability plot created."
    )

except Exception as e:
    print("\n❌ STEP 58 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 59: SINGLE MODEL VS REPEATED SHAP RANKING
# ============================================================

try:
    # --------------------------------------------------------
    # Original SHAP rank
    # --------------------------------------------------------
    original_rank = (
        shap_feature_importance[
            [
                "Feature",
                "Rank"
            ]
        ]
        .rename(
            columns={
                "Rank":
                    "Original_SHAP_Rank"
            }
        )
    )

    # --------------------------------------------------------
    # Stability rank
    # --------------------------------------------------------
    repeated_rank = (
        shap_stability_summary[
            [
                "Feature",
                "Median_Rank",
                "Mean_Rank",
                "Top_3_Frequency_Percent",
                "Top_5_Frequency_Percent"
            ]
        ]
    )

    shap_rank_comparison = (
        original_rank
        .merge(
            repeated_rank,
            on="Feature",
            how="inner"
        )
    )

    # --------------------------------------------------------
    # Spearman rank consistency
    # --------------------------------------------------------
    from scipy.stats import spearmanr

    rank_correlation = spearmanr(

        shap_rank_comparison[
            "Original_SHAP_Rank"
        ],

        shap_rank_comparison[
            "Mean_Rank"
        ]
    )

    print("=" * 90)
    print("ORIGINAL VS REPEATED SHAP RANKING")
    print("=" * 90)

    print(
        f"\nSpearman rank correlation: "
        f"{rank_correlation.statistic:.4f}"
    )

    print(
        f"p-value                  : "
        f"{rank_correlation.pvalue:.6g}"
    )

    display(
        shap_rank_comparison
        .sort_values(
            "Mean_Rank"
        )
        .head(15)
        .round(4)
    )

    print(
        "\n✅ Original and repeated SHAP rankings compared."
    )

except Exception as e:
    print("\n❌ STEP 59 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 60: SAVE FEATURE IMPORTANCE STABILITY RESULTS
# ============================================================

try:
    stability_files = {

        "shap_stability_fold_importance.csv":
            shap_stability_fold_importance,

        "shap_stability_summary.csv":
            shap_stability_summary,

        "shap_group_stability_folds.csv":
            shap_group_stability_folds,

        "shap_group_stability_summary.csv":
            shap_group_stability_summary,

        "shap_rank_comparison.csv":
            shap_rank_comparison
    }

    for filename, dataframe in (
        stability_files.items()
    ):

        filepath = (
            f"{OUT_DIR}/"
            f"{filename}"
        )

        dataframe.to_csv(
            filepath,
            index=False
        )

        print(
            f"✅ Saved: {filename}"
        )

    print(
        "\n✅ All feature-stability results saved."
    )

except Exception as e:
    print("\n❌ STEP 60 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 16. Calibration Analysis

Discrimination and calibration evaluate different aspects of predictive performance.

ROC-AUC measures the ability of a model to distinguish between Dry Eye and non-Dry-Eye observations, whereas calibration evaluates whether predicted probabilities correspond to observed outcome frequencies.

Calibration is assessed using out-of-fold predictions from the training dataset so that each probability is generated for an observation that was not used to fit the corresponding model.

The following measures are reported:

- Brier Score
- Log Loss
- Expected Calibration Error (ECE)
- Mean predicted probability
- Observed Dry Eye prevalence
- ROC-AUC
- PR-AUC

Lower values of Brier Score, Log Loss, and ECE indicate better probability calibration.

The held-out final test set remains untouched.

In [ ]:
# ============================================================
# STEP 61: GENERATE OUT-OF-FOLD PROBABILITY PREDICTIONS
# ============================================================

try:
    import numpy as np
    import pandas as pd

    from sklearn.model_selection import cross_val_predict
    from sklearn.base import clone

    print("=" * 85)
    print("OUT-OF-FOLD CALIBRATION PREDICTIONS")
    print("=" * 85)

    # --------------------------------------------------------
    # Full-feature models used for calibration assessment
    # --------------------------------------------------------
    CALIBRATION_MODELS = {

        "Logistic Regression":
            logistic_model,

        "XGBoost":
            xgboost_model
    }

    # --------------------------------------------------------
    # Use all 26 predictors
    # --------------------------------------------------------
    X_calibration = X_train[
        ABLATION_CONFIGS["All Features"]
    ].copy()

    y_calibration = y_train.copy()

    calibration_prediction_rows = []

    calibration_probabilities = {}

    for model_name, model in (
        CALIBRATION_MODELS.items()
    ):

        print(
            f"\nGenerating OOF probabilities: "
            f"{model_name}..."
        )

        # ----------------------------------------------------
        # Each observation receives a probability from
        # a model that did not train on that observation.
        # ----------------------------------------------------
        oof_probability = cross_val_predict(

            estimator=clone(model),

            X=X_calibration,

            y=y_calibration,

            cv=cv5,

            method="predict_proba",

            n_jobs=1
        )[:, 1]

        calibration_probabilities[
            model_name
        ] = oof_probability

        for observation_index, (
            true_label,
            probability
        ) in enumerate(
            zip(
                y_calibration.to_numpy(),
                oof_probability
            )
        ):

            calibration_prediction_rows.append({

                "Model":
                    model_name,

                "Observation":
                    observation_index,

                "True_Label":
                    int(true_label),

                "Predicted_Probability":
                    float(probability)
            })

        print(
            f"✅ {model_name} completed."
        )

        print(
            f"   Mean predicted probability: "
            f"{oof_probability.mean():.4f}"
        )

    calibration_oof_predictions = pd.DataFrame(
        calibration_prediction_rows
    )

    print(
        "\nObserved Dry Eye prevalence: "
        f"{y_calibration.mean():.4f}"
    )

    print(
        "\n✅ Out-of-fold probabilities generated."
    )

except Exception as e:
    print("\n❌ STEP 61 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 16.1 Expected Calibration Error

Expected Calibration Error (ECE) divides predicted probabilities into probability intervals and compares average predicted probability with the actual outcome frequency within each interval.

The differences are weighted by the number of observations in each interval.

An ECE closer to zero indicates better agreement between predicted probabilities and observed outcomes.

In [ ]:
# ============================================================
# STEP 62: EXPECTED CALIBRATION ERROR FUNCTION
# ============================================================

try:
    def expected_calibration_error(
        y_true,
        y_prob,
        n_bins=10
    ):
        """
        Calculate Expected Calibration Error (ECE)
        using equally spaced probability bins.
        """

        y_true = np.asarray(
            y_true
        )

        y_prob = np.asarray(
            y_prob
        )

        bin_edges = np.linspace(
            0.0,
            1.0,
            n_bins + 1
        )

        # Produces bin IDs from 0 to n_bins-1
        bin_ids = np.digitize(
            y_prob,
            bin_edges[1:-1],
            right=False
        )

        ece = 0.0

        for bin_id in range(
            n_bins
        ):

            mask = (
                bin_ids
                ==
                bin_id
            )

            if not np.any(mask):
                continue

            observed_frequency = (
                y_true[mask]
                .mean()
            )

            mean_probability = (
                y_prob[mask]
                .mean()
            )

            bin_weight = (
                mask.mean()
            )

            ece += (
                bin_weight
                *
                abs(
                    observed_frequency
                    -
                    mean_probability
                )
            )

        return float(ece)

    print(
        "✅ Expected Calibration Error function created."
    )

except Exception as e:
    print("\n❌ STEP 62 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 16.2 Calibration Metrics

Calibration performance is quantified using Brier Score, Log Loss, and Expected Calibration Error.

The Brier Score measures the mean squared difference between predicted probabilities and observed binary outcomes.

Log Loss penalizes incorrect high-confidence predictions.

For both metrics, lower values indicate better probabilistic performance.

In [ ]:
# ============================================================
# STEP 63: CALCULATE CALIBRATION METRICS
# CHECKPOINT 8A
# ============================================================

try:
    from sklearn.metrics import (
        brier_score_loss,
        log_loss,
        roc_auc_score,
        average_precision_score
    )

    calibration_summary_rows = []

    observed_prevalence = (
        y_calibration.mean()
    )

    for model_name in (
        CALIBRATION_MODELS.keys()
    ):

        probabilities = (
            calibration_probabilities[
                model_name
            ]
        )

        # ----------------------------------------------------
        # Calibration metrics
        # ----------------------------------------------------
        brier = brier_score_loss(
            y_calibration,
            probabilities
        )

        logloss = log_loss(
            y_calibration,
            probabilities
        )

        ece = expected_calibration_error(
            y_calibration,
            probabilities,
            n_bins=10
        )

        # ----------------------------------------------------
        # Discrimination metrics included for context
        # ----------------------------------------------------
        roc_auc = roc_auc_score(
            y_calibration,
            probabilities
        )

        pr_auc = average_precision_score(
            y_calibration,
            probabilities
        )

        mean_probability = (
            probabilities.mean()
        )

        probability_bias = (
            mean_probability
            -
            observed_prevalence
        )

        calibration_summary_rows.append({

            "Model":
                model_name,

            "Observed_Prevalence":
                observed_prevalence,

            "Mean_Predicted_Probability":
                mean_probability,

            "Probability_Bias":
                probability_bias,

            "Brier_Score":
                brier,

            "Log_Loss":
                logloss,

            "ECE_10_Bins":
                ece,

            "ROC_AUC":
                roc_auc,

            "PR_AUC":
                pr_auc
        })

    calibration_summary = pd.DataFrame(
        calibration_summary_rows
    )

    print("=" * 100)
    print("CALIBRATION SUMMARY")
    print("=" * 100)

    display(
        calibration_summary
        .round(5)
    )

    print(
        "\nInterpretation:"
    )

    print(
        "• Brier Score: lower is better"
    )

    print(
        "• Log Loss: lower is better"
    )

    print(
        "• ECE: closer to 0 is better"
    )

    print(
        "• Probability Bias: closer to 0 is better"
    )

    print(
        "\n✅ CHECKPOINT 8A PASSED."
    )

except Exception as e:
    print("\n❌ CHECKPOINT 8A FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 16.3 Prevalence-Only Probability Reference

A prevalence-only reference model assigns every observation the same probability equal to the Dry Eye prevalence in the training data.

This reference has no discriminatory ability but provides an important benchmark for probabilistic metrics such as Brier Score and Log Loss.

A useful predictive model should ideally improve probabilistic accuracy relative to this reference while also providing discrimination above ROC-AUC = 0.50.

In [ ]:
# ============================================================
# STEP 64: PREVALENCE-ONLY PROBABILITY BASELINE
# ============================================================

try:
    prevalence_probability = np.full(
        shape=len(y_calibration),
        fill_value=observed_prevalence,
        dtype=float
    )

    prevalence_brier = brier_score_loss(
        y_calibration,
        prevalence_probability
    )

    prevalence_logloss = log_loss(
        y_calibration,
        prevalence_probability
    )

    prevalence_ece = expected_calibration_error(
        y_calibration,
        prevalence_probability,
        n_bins=10
    )

    prevalence_reference = pd.DataFrame([{

        "Model":
            "Prevalence-Only Reference",

        "Observed_Prevalence":
            observed_prevalence,

        "Mean_Predicted_Probability":
            observed_prevalence,

        "Probability_Bias":
            0.0,

        "Brier_Score":
            prevalence_brier,

        "Log_Loss":
            prevalence_logloss,

        "ECE_10_Bins":
            prevalence_ece,

        "ROC_AUC":
            0.5,

        "PR_AUC":
            observed_prevalence
    }])

    calibration_comparison = pd.concat(
        [
            calibration_summary,
            prevalence_reference
        ],
        ignore_index=True
    )

    print("=" * 100)
    print("CALIBRATION VS PREVALENCE-ONLY REFERENCE")
    print("=" * 100)

    display(
        calibration_comparison
        .round(5)
    )

    print(
        "\n✅ Probability reference comparison completed."
    )

except Exception as e:
    print("\n❌ STEP 64 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 16.4 Calibration Curves

Calibration curves compare average predicted probability with the actual frequency of Dry Eye Disease.

A perfectly calibrated model would follow the 45-degree reference line.

Predictions above or below this line indicate systematic underestimation or overestimation of observed risk.

In [ ]:
# ============================================================
# STEP 65: CREATE CALIBRATION CURVE POINTS
# ============================================================

try:
    from sklearn.calibration import calibration_curve

    calibration_curve_rows = []

    for model_name in (
        CALIBRATION_MODELS.keys()
    ):

        probabilities = (
            calibration_probabilities[
                model_name
            ]
        )

        # ----------------------------------------------------
        # Quantile bins give approximately equal numbers
        # of observations per calibration point.
        # ----------------------------------------------------
        observed_fraction, mean_prediction = (
            calibration_curve(
                y_true=y_calibration,
                y_prob=probabilities,
                n_bins=10,
                strategy="quantile"
            )
        )

        for bin_number, (
            pred,
            observed
        ) in enumerate(
            zip(
                mean_prediction,
                observed_fraction
            ),
            start=1
        ):

            calibration_curve_rows.append({

                "Model":
                    model_name,

                "Bin":
                    bin_number,

                "Mean_Predicted_Probability":
                    pred,

                "Observed_Fraction_Positive":
                    observed
            })

    calibration_curve_points = pd.DataFrame(
        calibration_curve_rows
    )

    print("=" * 90)
    print("CALIBRATION CURVE POINTS")
    print("=" * 90)

    display(
        calibration_curve_points
        .round(4)
    )

    print(
        "\n✅ Calibration curve points created."
    )

except Exception as e:
    print("\n❌ STEP 65 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 66: PLOT CALIBRATION CURVES
# ============================================================

try:
    plt.figure(
        figsize=(8, 6)
    )

    # --------------------------------------------------------
    # Perfect calibration reference
    # --------------------------------------------------------
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect Calibration"
    )

    # --------------------------------------------------------
    # Plot each model
    # --------------------------------------------------------
    for model_name in (
        CALIBRATION_MODELS.keys()
    ):

        temp = (
            calibration_curve_points[
                calibration_curve_points[
                    "Model"
                ]
                ==
                model_name
            ]
        )

        plt.plot(
            temp[
                "Mean_Predicted_Probability"
            ],

            temp[
                "Observed_Fraction_Positive"
            ],

            marker="o",

            label=model_name
        )

    plt.xlabel(
        "Mean Predicted Probability"
    )

    plt.ylabel(
        "Observed Dry Eye Frequency"
    )

    plt.title(
        "Out-of-Fold Calibration Curves"
    )

    plt.legend()

    plt.tight_layout()
    plt.show()

    print(
        "✅ Calibration curve created."
    )

except Exception as e:
    print("\n❌ STEP 66 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 16.5 Predicted Probability Distribution

The distribution of out-of-fold predicted probabilities is examined separately for Dry Eye and non-Dry-Eye observations.

Large overlap between the two distributions would be consistent with the modest ROC-AUC values observed earlier.### 16.5 Predicted Probability Distribution

The distribution of out-of-fold predicted probabilities is examined separately for Dry Eye and non-Dry-Eye observations.

Large overlap between the two distributions would be consistent with the modest ROC-AUC values observed earlier.

In [ ]:
# ============================================================
# STEP 67: PREDICTED PROBABILITY DISTRIBUTIONS
# ============================================================

try:
    for model_name in (
        CALIBRATION_MODELS.keys()
    ):

        temp = (
            calibration_oof_predictions[
                calibration_oof_predictions[
                    "Model"
                ]
                ==
                model_name
            ]
            .copy()
        )

        temp[
            "Dry Eye Status"
        ] = (
            temp[
                "True_Label"
            ]
            .map({
                0: "No Dry Eye",
                1: "Dry Eye"
            })
        )

        plt.figure(
            figsize=(8, 5)
        )

        sns.histplot(
            data=temp,
            x="Predicted_Probability",
            hue="Dry Eye Status",
            bins=30,
            stat="density",
            common_norm=False,
            element="step",
            fill=False
        )

        plt.axvline(
            0.5,
            linestyle="--",
            linewidth=1
        )

        plt.xlabel(
            "Predicted Probability of Dry Eye"
        )

        plt.ylabel(
            "Density"
        )

        plt.title(
            f"OOF Probability Distribution — {model_name}"
        )

        plt.tight_layout()
        plt.show()

    print(
        "✅ Probability distributions created."
    )

except Exception as e:
    print("\n❌ STEP 67 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))# ============================================================
# STEP 67: PREDICTED PROBABILITY DISTRIBUTIONS
# ============================================================

try:
    for model_name in (
        CALIBRATION_MODELS.keys()
    ):

        temp = (
            calibration_oof_predictions[
                calibration_oof_predictions[
                    "Model"
                ]
                ==
                model_name
            ]
            .copy()
        )

        temp[
            "Dry Eye Status"
        ] = (
            temp[
                "True_Label"
            ]
            .map({
                0: "No Dry Eye",
                1: "Dry Eye"
            })
        )

        plt.figure(
            figsize=(8, 5)
        )

        sns.histplot(
            data=temp,
            x="Predicted_Probability",
            hue="Dry Eye Status",
            bins=30,
            stat="density",
            common_norm=False,
            element="step",
            fill=False
        )

        plt.axvline(
            0.5,
            linestyle="--",
            linewidth=1
        )

        plt.xlabel(
            "Predicted Probability of Dry Eye"
        )

        plt.ylabel(
            "Density"
        )

        plt.title(
            f"OOF Probability Distribution — {model_name}"
        )

        plt.tight_layout()
        plt.show()

    print(
        "✅ Probability distributions created."
    )

except Exception as e:
    print("\n❌ STEP 67 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 68: BIN-LEVEL CALIBRATION TABLE
# ============================================================

try:
    bin_calibration_rows = []

    for model_name in (
        CALIBRATION_MODELS.keys()
    ):

        probabilities = (
            calibration_probabilities[
                model_name
            ]
        )

        temp = pd.DataFrame({

            "True_Label":
                y_calibration.to_numpy(),

            "Predicted_Probability":
                probabilities
        })

        # ----------------------------------------------------
        # Quantile bins: approximately equal number of
        # observations in each group.
        # ----------------------------------------------------
        temp["Probability_Bin"] = pd.qcut(

            temp[
                "Predicted_Probability"
            ],

            q=10,

            duplicates="drop"
        )

        grouped = (
            temp
            .groupby(
                "Probability_Bin",
                observed=True
            )
            .agg(

                Count=(
                    "True_Label",
                    "size"
                ),

                Mean_Predicted_Probability=(
                    "Predicted_Probability",
                    "mean"
                ),

                Observed_Dry_Eye_Rate=(
                    "True_Label",
                    "mean"
                )
            )
            .reset_index()
        )

        grouped[
            "Absolute_Calibration_Error"
        ] = (

            grouped[
                "Mean_Predicted_Probability"
            ]

            -

            grouped[
                "Observed_Dry_Eye_Rate"
            ]

        ).abs()

        grouped[
            "Model"
        ] = model_name

        bin_calibration_rows.append(
            grouped
        )

    bin_calibration_table = pd.concat(
        bin_calibration_rows,
        ignore_index=True
    )

    print("=" * 100)
    print("BIN-LEVEL CALIBRATION TABLE")
    print("=" * 100)

    display(
        bin_calibration_table[
            [
                "Model",
                "Probability_Bin",
                "Count",
                "Mean_Predicted_Probability",
                "Observed_Dry_Eye_Rate",
                "Absolute_Calibration_Error"
            ]
        ].round(4)
    )

    print(
        "\n✅ Bin-level calibration table completed."
    )

except Exception as e:
    print("\n❌ STEP 68 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 69: CALIBRATION INTERPRETATION SUMMARY
# ============================================================

try:
    print("=" * 90)
    print("CALIBRATION INTERPRETATION")
    print("=" * 90)

    reference_brier = (
        prevalence_reference[
            "Brier_Score"
        ]
        .iloc[0]
    )

    for _, row in (
        calibration_summary.iterrows()
    ):

        print(
            f"\n{row['Model']}"
        )

        print("-" * 60)

        print(
            f"Observed prevalence        : "
            f"{row['Observed_Prevalence']:.4f}"
        )

        print(
            f"Mean predicted probability : "
            f"{row['Mean_Predicted_Probability']:.4f}"
        )

        print(
            f"Probability bias           : "
            f"{row['Probability_Bias']:+.4f}"
        )

        print(
            f"Brier Score                : "
            f"{row['Brier_Score']:.4f}"
        )

        print(
            f"ECE                        : "
            f"{row['ECE_10_Bins']:.4f}"
        )

        brier_improvement = (
            reference_brier
            -
            row["Brier_Score"]
        )

        print(
            f"Brier improvement vs "
            f"prevalence reference       : "
            f"{brier_improvement:+.4f}"
        )

    print(
        "\nNOTE:"
    )

    print(
        "Calibration results describe probability reliability. "
        "They do not change the previously observed "
        "ROC-AUC discrimination results."
    )

    print(
        "\n✅ Calibration interpretation completed."
    )

except Exception as e:
    print("\n❌ STEP 69 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 70: SAVE CALIBRATION RESULTS
# ============================================================

try:
    calibration_files = {

        "calibration_oof_predictions.csv":
            calibration_oof_predictions,

        "calibration_summary.csv":
            calibration_summary,

        "calibration_comparison.csv":
            calibration_comparison,

        "calibration_curve_points.csv":
            calibration_curve_points,

        "calibration_bin_table.csv":
            bin_calibration_table
    }

    for filename, dataframe in (
        calibration_files.items()
    ):

        filepath = (
            f"{OUT_DIR}/"
            f"{filename}"
        )

        dataframe.to_csv(
            filepath,
            index=False
        )

        print(
            f"✅ Saved: {filename}"
        )

    print(
        "\n✅ All calibration results saved successfully."
    )

except Exception as e:
    print("\n❌ STEP 70 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 17. Final Held-Out Test Evaluation

All model development, feature-group comparisons, statistical analyses, explainability analyses, and calibration assessments were performed using the training dataset.

The held-out 20% test set is now used for the first and final evaluation.

No model, feature group, hyperparameter, or classification threshold will be changed based on test-set performance.

Six pre-specified model/configuration combinations are evaluated:

### Logistic Regression
- All Features
- No Ocular Symptoms
- Ocular Symptoms Only

### XGBoost
- All Features
- No Ocular Symptoms
- Ocular Symptoms Only

The primary held-out metric is ROC-AUC.

Additional metrics include:

- PR-AUC
- Accuracy
- Balanced Accuracy
- Precision
- Sensitivity / Recall
- Specificity
- F1 Score
- Brier Score
- Log Loss

The default 0.50 probability threshold is retained for threshold-dependent metrics.

In [ ]:
# ============================================================
# STEP 71: FINAL TEST-SET SAFETY CHECK
# ============================================================

try:
    print("=" * 85)
    print("FINAL TEST-SET SAFETY CHECK")
    print("=" * 85)

    print(f"\nTraining observations : {len(X_train):,}")
    print(f"Test observations     : {len(X_test):,}")

    print(f"Training predictors   : {X_train.shape[1]}")
    print(f"Test predictors       : {X_test.shape[1]}")

    # --------------------------------------------------------
    # Basic integrity checks
    # --------------------------------------------------------
    if "Dry_Eye" in X_train.columns:
        raise ValueError(
            "Target leakage detected in X_train."
        )

    if "Dry_Eye" in X_test.columns:
        raise ValueError(
            "Target leakage detected in X_test."
        )

    overlap = set(X_train.index).intersection(
        set(X_test.index)
    )

    if len(overlap) != 0:
        raise ValueError(
            f"Train/test overlap detected: {len(overlap)} rows."
        )

    if list(X_train.columns) != list(X_test.columns):
        raise ValueError(
            "Train/test feature columns do not match."
        )

    print("\n✅ No target leakage.")
    print("✅ No train/test overlap.")
    print("✅ Feature columns match.")
    print("✅ Final test evaluation may proceed.")

except Exception as e:
    print("\n❌ STEP 71 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 17.1 Final Model Evaluation

Each pre-specified model is refitted using the complete training dataset and then evaluated once on the held-out test set.

The test set is not used for model fitting, model selection, feature selection, or threshold optimization.

In [ ]:
# ============================================================
# STEP 72: FINAL HELD-OUT TEST EVALUATION
# ============================================================

try:
    import time
    import numpy as np
    import pandas as pd

    from sklearn.base import clone

    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        log_loss,
        confusion_matrix
    )

    print("=" * 95)
    print("FINAL HELD-OUT TEST EVALUATION")
    print("=" * 95)

    # --------------------------------------------------------
    # Frozen configurations
    # --------------------------------------------------------
    FINAL_CONFIGS = {

        "All Features":
            ABLATION_CONFIGS["All Features"],

        "No Ocular Symptoms":
            ABLATION_CONFIGS["No Ocular Symptoms"],

        "Ocular Symptoms Only":
            ABLATION_CONFIGS["Ocular Symptoms Only"]
    }

    FINAL_MODELS = {

        "Logistic Regression":
            logistic_model,

        "XGBoost":
            xgboost_model
    }

    final_result_rows = []
    final_prediction_rows = []

    # Save fitted models for later plots
    final_fitted_models = {}

    for model_name, model in FINAL_MODELS.items():

        print("\n" + "-" * 95)
        print(model_name)
        print("-" * 95)

        for config_name, feature_list in FINAL_CONFIGS.items():

            print(
                f"Evaluating {config_name}...",
                end=" "
            )

            # ------------------------------------------------
            # Select the pre-defined feature configuration
            # ------------------------------------------------
            Xtr = X_train[
                feature_list
            ].copy()

            Xte = X_test[
                feature_list
            ].copy()

            # ------------------------------------------------
            # Fit a fresh model using ALL training observations
            # ------------------------------------------------
            final_model = clone(
                model
            )

            start_time = time.time()

            final_model.fit(
                Xtr,
                y_train
            )

            fit_time = (
                time.time()
                -
                start_time
            )

            # ------------------------------------------------
            # FINAL TEST PREDICTIONS
            # ------------------------------------------------
            test_probability = (
                final_model
                .predict_proba(
                    Xte
                )[:, 1]
            )

            test_prediction = (
                test_probability
                >=
                0.50
            ).astype(int)

            # ------------------------------------------------
            # Confusion matrix
            # ------------------------------------------------
            tn, fp, fn, tp = (
                confusion_matrix(
                    y_test,
                    test_prediction,
                    labels=[0, 1]
                )
                .ravel()
            )

            specificity = (
                tn / (tn + fp)
                if (tn + fp) > 0
                else np.nan
            )

            # ------------------------------------------------
            # Metrics
            # ------------------------------------------------
            result = {

                "Model":
                    model_name,

                "Configuration":
                    config_name,

                "N_Features":
                    len(feature_list),

                "Accuracy":
                    accuracy_score(
                        y_test,
                        test_prediction
                    ),

                "Balanced_Accuracy":
                    balanced_accuracy_score(
                        y_test,
                        test_prediction
                    ),

                "Precision":
                    precision_score(
                        y_test,
                        test_prediction,
                        zero_division=0
                    ),

                "Sensitivity_Recall":
                    recall_score(
                        y_test,
                        test_prediction,
                        zero_division=0
                    ),

                "Specificity":
                    specificity,

                "F1":
                    f1_score(
                        y_test,
                        test_prediction,
                        zero_division=0
                    ),

                "ROC_AUC":
                    roc_auc_score(
                        y_test,
                        test_probability
                    ),

                "PR_AUC":
                    average_precision_score(
                        y_test,
                        test_probability
                    ),

                "Brier_Score":
                    brier_score_loss(
                        y_test,
                        test_probability
                    ),

                "Log_Loss":
                    log_loss(
                        y_test,
                        test_probability
                    ),

                "TN":
                    tn,

                "FP":
                    fp,

                "FN":
                    fn,

                "TP":
                    tp,

                "Fit_Time_Seconds":
                    fit_time
            }

            final_result_rows.append(
                result
            )

            # ------------------------------------------------
            # Save individual test predictions
            # ------------------------------------------------
            for test_index, true_value, pred_value, probability in zip(

                Xte.index,

                y_test.to_numpy(),

                test_prediction,

                test_probability
            ):

                final_prediction_rows.append({

                    "Test_Index":
                        int(test_index),

                    "Model":
                        model_name,

                    "Configuration":
                        config_name,

                    "True_Label":
                        int(true_value),

                    "Predicted_Label":
                        int(pred_value),

                    "Predicted_Probability":
                        float(probability)
                })

            final_fitted_models[
                (
                    model_name,
                    config_name
                )
            ] = final_model

            print(
                f"✅ ROC-AUC = "
                f"{result['ROC_AUC']:.4f}"
            )

    # --------------------------------------------------------
    # Final result tables
    # --------------------------------------------------------
    final_test_results = pd.DataFrame(
        final_result_rows
    )

    final_test_predictions = pd.DataFrame(
        final_prediction_rows
    )

    print("\n" + "=" * 95)
    print("FINAL HELD-OUT TEST RESULTS")
    print("=" * 95)

    display_columns = [
        "Model",
        "Configuration",
        "N_Features",
        "Accuracy",
        "Balanced_Accuracy",
        "Precision",
        "Sensitivity_Recall",
        "Specificity",
        "F1",
        "ROC_AUC",
        "PR_AUC",
        "Brier_Score"
    ]

    display(
        final_test_results[
            display_columns
        ].round(4)
    )

    print("\n✅ FINAL TEST EVALUATION COMPLETED.")
    print("✅ Test set has now been used exactly once.")

except Exception as e:
    print("\n❌ STEP 72 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### >> > > > > > > > >  **>**>  17.2 Final Held-Out Performance

The final test results are summarized using the untouched held-out observations.

Because the study focuses on the source of predictive discrimination, particular emphasis is placed on ROC-AUC differences between the full-feature, non-ocular, and ocular-only configurations.

In [ ]:
# ============================================================
# STEP 73: PUBLICATION-READY FINAL TEST TABLE
# ============================================================

try:
    paper_final_table = (
        final_test_results[
            [
                "Model",
                "Configuration",
                "N_Features",
                "Accuracy",
                "Balanced_Accuracy",
                "Precision",
                "Sensitivity_Recall",
                "Specificity",
                "F1",
                "ROC_AUC",
                "PR_AUC",
                "Brier_Score"
            ]
        ]
        .copy()
    )

    numeric_columns = [
        "Accuracy",
        "Balanced_Accuracy",
        "Precision",
        "Sensitivity_Recall",
        "Specificity",
        "F1",
        "ROC_AUC",
        "PR_AUC",
        "Brier_Score"
    ]

    paper_final_table[
        numeric_columns
    ] = (
        paper_final_table[
            numeric_columns
        ]
        .round(3)
    )

    print("=" * 100)
    print("TABLE: FINAL HELD-OUT TEST PERFORMANCE")
    print("=" * 100)

    display(
        paper_final_table
    )

    print(
        "\n✅ Final publication table created."
    )

except Exception as e:
    print("\n❌ STEP 73 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 17.3 Held-Out ROC Curves

ROC curves are generated from the final held-out test probabilities.

The curves provide a visual comparison of discrimination between:

- the full predictor set;
- the predictor set without ocular symptoms;
- the three ocular symptoms alone.

In [ ]:
# ============================================================
# STEP 74: FINAL TEST ROC CURVES
# ============================================================

try:
    from sklearn.metrics import roc_curve

    for model_name in FINAL_MODELS.keys():

        plt.figure(
            figsize=(8, 6)
        )

        for config_name in FINAL_CONFIGS.keys():

            temp = final_test_predictions[
                (
                    final_test_predictions["Model"]
                    ==
                    model_name
                )
                &
                (
                    final_test_predictions[
                        "Configuration"
                    ]
                    ==
                    config_name
                )
            ]

            fpr, tpr, _ = roc_curve(

                temp["True_Label"],

                temp[
                    "Predicted_Probability"
                ]
            )

            auc_value = roc_auc_score(

                temp["True_Label"],

                temp[
                    "Predicted_Probability"
                ]
            )

            plt.plot(
                fpr,
                tpr,
                linewidth=2,
                label=(
                    f"{config_name} "
                    f"(AUC={auc_value:.3f})"
                )
            )

        # Chance reference
        plt.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Chance"
        )

        plt.xlabel(
            "False Positive Rate"
        )

        plt.ylabel(
            "True Positive Rate"
        )

        plt.title(
            f"Held-Out ROC Curves — {model_name}"
        )

        plt.legend(
            loc="lower right"
        )

        plt.tight_layout()

        filename = (
            f"{OUT_DIR}/"
            +
            model_name
            .lower()
            .replace(" ", "_")
            +
            "_final_roc.png"
        )

        plt.savefig(
            filename,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()

    print(
        "✅ Final ROC figures created and saved."
    )

except Exception as e:
    print("\n❌ STEP 74 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 75: FINAL TEST PRECISION-RECALL CURVES
# ============================================================

try:
    from sklearn.metrics import precision_recall_curve

    for model_name in FINAL_MODELS.keys():

        plt.figure(
            figsize=(8, 6)
        )

        for config_name in FINAL_CONFIGS.keys():

            temp = final_test_predictions[
                (
                    final_test_predictions["Model"]
                    ==
                    model_name
                )
                &
                (
                    final_test_predictions[
                        "Configuration"
                    ]
                    ==
                    config_name
                )
            ]

            precision_values, recall_values, _ = (
                precision_recall_curve(

                    temp["True_Label"],

                    temp[
                        "Predicted_Probability"
                    ]
                )
            )

            pr_auc = (
                average_precision_score(

                    temp["True_Label"],

                    temp[
                        "Predicted_Probability"
                    ]
                )
            )

            plt.plot(
                recall_values,
                precision_values,
                linewidth=2,
                label=(
                    f"{config_name} "
                    f"(AP={pr_auc:.3f})"
                )
            )

        # Prevalence reference
        plt.axhline(
            y=y_test.mean(),
            linestyle="--",
            label=(
                f"Prevalence = "
                f"{y_test.mean():.3f}"
            )
        )

        plt.xlabel(
            "Recall"
        )

        plt.ylabel(
            "Precision"
        )

        plt.title(
            f"Held-Out Precision-Recall Curves — {model_name}"
        )

        plt.legend()

        plt.tight_layout()

        filename = (
            f"{OUT_DIR}/"
            +
            model_name
            .lower()
            .replace(" ", "_")
            +
            "_final_pr_curve.png"
        )

        plt.savefig(
            filename,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()

    print(
        "✅ Final precision-recall figures saved."
    )

except Exception as e:
    print("\n❌ STEP 75 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 17.4 Classification Behavior at the Default Threshold

Confusion matrices are presented for the complete Logistic Regression and XGBoost models using the unchanged 0.50 probability threshold.

These matrices illustrate the trade-off between sensitivity and specificity.

In [ ]:
# ============================================================
# STEP 76: FINAL CONFUSION MATRICES — ALL FEATURES
# ============================================================

try:
    from sklearn.metrics import ConfusionMatrixDisplay

    for model_name in FINAL_MODELS.keys():

        temp = final_test_predictions[
            (
                final_test_predictions["Model"]
                ==
                model_name
            )
            &
            (
                final_test_predictions[
                    "Configuration"
                ]
                ==
                "All Features"
            )
        ]

        cm = confusion_matrix(
            temp["True_Label"],
            temp["Predicted_Label"],
            labels=[0, 1]
        )

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=[
                "No Dry Eye",
                "Dry Eye"
            ]
        )

        fig, ax = plt.subplots(
            figsize=(6, 5)
        )

        disp.plot(
            ax=ax,
            cmap=None,
            colorbar=False
        )

        plt.title(
            f"Held-Out Confusion Matrix — {model_name}"
        )

        plt.tight_layout()

        filename = (
            f"{OUT_DIR}/"
            +
            model_name
            .lower()
            .replace(" ", "_")
            +
            "_confusion_matrix.png"
        )

        plt.savefig(
            filename,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()

    print(
        "✅ Final confusion matrices created."
    )

except Exception as e:
    print("\n❌ STEP 76 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 18. Held-Out Performance Uncertainty

Bootstrap resampling of the held-out test observations is used to estimate 95% confidence intervals for ROC-AUC.

This analysis is performed only after all modeling decisions have been finalized.

The intervals quantify uncertainty around the final test-set discrimination estimates and are not used for further model selection.

In [ ]:
# ============================================================
# STEP 77: BOOTSTRAP 95% CIs FOR FINAL TEST ROC-AUC
# ============================================================

try:
    rng = np.random.default_rng(
        RANDOM_STATE
    )

    N_TEST_BOOTSTRAPS = 2000

    test_auc_ci_rows = []

    print("=" * 90)
    print("BOOTSTRAP HELD-OUT ROC-AUC INTERVALS")
    print("=" * 90)

    for model_name in FINAL_MODELS.keys():

        for config_name in FINAL_CONFIGS.keys():

            temp = (
                final_test_predictions[
                    (
                        final_test_predictions[
                            "Model"
                        ]
                        ==
                        model_name
                    )
                    &
                    (
                        final_test_predictions[
                            "Configuration"
                        ]
                        ==
                        config_name
                    )
                ]
                .reset_index(drop=True)
            )

            y_true_array = (
                temp["True_Label"]
                .to_numpy()
            )

            y_prob_array = (
                temp[
                    "Predicted_Probability"
                ]
                .to_numpy()
            )

            observed_auc = roc_auc_score(
                y_true_array,
                y_prob_array
            )

            bootstrap_aucs = []

            n_test = len(
                y_true_array
            )

            for _ in range(
                N_TEST_BOOTSTRAPS
            ):

                sample_indices = rng.integers(
                    0,
                    n_test,
                    size=n_test
                )

                y_boot = (
                    y_true_array[
                        sample_indices
                    ]
                )

                p_boot = (
                    y_prob_array[
                        sample_indices
                    ]
                )

                # Skip rare invalid bootstrap samples
                # containing only one class.
                if (
                    len(
                        np.unique(
                            y_boot
                        )
                    )
                    <
                    2
                ):
                    continue

                bootstrap_aucs.append(
                    roc_auc_score(
                        y_boot,
                        p_boot
                    )
                )

            bootstrap_aucs = np.array(
                bootstrap_aucs
            )

            lower = np.percentile(
                bootstrap_aucs,
                2.5
            )

            upper = np.percentile(
                bootstrap_aucs,
                97.5
            )

            test_auc_ci_rows.append({

                "Model":
                    model_name,

                "Configuration":
                    config_name,

                "ROC_AUC":
                    observed_auc,

                "CI_95_Lower":
                    lower,

                "CI_95_Upper":
                    upper,

                "Bootstrap_Samples":
                    len(
                        bootstrap_aucs
                    )
            })

    final_auc_confidence_intervals = pd.DataFrame(
        test_auc_ci_rows
    )

    display(
        final_auc_confidence_intervals
        .round(4)
    )

    print(
        "\n✅ Final ROC-AUC confidence intervals calculated."
    )

except Exception as e:
    print("\n❌ STEP 77 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

### 18.1 Held-Out Feature-Group Differences

Paired bootstrap resampling is used to quantify the held-out ROC-AUC difference between feature configurations.

Because probabilities from the competing models correspond to the same test observations, identical bootstrap samples are used for both configurations.

The main comparisons are:

1. All Features vs No Ocular Symptoms
2. Ocular Symptoms Only vs No Ocular Symptoms
3. All Features vs Ocular Symptoms Only

In [ ]:
# ============================================================
# STEP 78: PAIRED BOOTSTRAP TEST-SET AUC DIFFERENCES
# ============================================================

try:
    rng = np.random.default_rng(
        RANDOM_STATE + 100
    )

    N_DELTA_BOOTSTRAPS = 2000

    FINAL_COMPARISONS = [

        (
            "All Features",
            "No Ocular Symptoms"
        ),

        (
            "Ocular Symptoms Only",
            "No Ocular Symptoms"
        ),

        (
            "All Features",
            "Ocular Symptoms Only"
        )
    ]

    final_delta_rows = []

    for model_name in FINAL_MODELS.keys():

        model_predictions = (
            final_test_predictions[
                final_test_predictions[
                    "Model"
                ]
                ==
                model_name
            ]
        )

        for config_a, config_b in FINAL_COMPARISONS:

            pred_a = (
                model_predictions[
                    model_predictions[
                        "Configuration"
                    ]
                    ==
                    config_a
                ]
                .sort_values(
                    "Test_Index"
                )
            )

            pred_b = (
                model_predictions[
                    model_predictions[
                        "Configuration"
                    ]
                    ==
                    config_b
                ]
                .sort_values(
                    "Test_Index"
                )
            )

            # ------------------------------------------------
            # Make sure rows are perfectly aligned
            # ------------------------------------------------
            if not np.array_equal(

                pred_a[
                    "Test_Index"
                ].to_numpy(),

                pred_b[
                    "Test_Index"
                ].to_numpy()
            ):
                raise ValueError(
                    "Test predictions are not aligned."
                )

            y_array = (
                pred_a[
                    "True_Label"
                ]
                .to_numpy()
            )

            prob_a = (
                pred_a[
                    "Predicted_Probability"
                ]
                .to_numpy()
            )

            prob_b = (
                pred_b[
                    "Predicted_Probability"
                ]
                .to_numpy()
            )

            auc_a = roc_auc_score(
                y_array,
                prob_a
            )

            auc_b = roc_auc_score(
                y_array,
                prob_b
            )

            observed_difference = (
                auc_a
                -
                auc_b
            )

            n = len(
                y_array
            )

            bootstrap_differences = []

            for _ in range(
                N_DELTA_BOOTSTRAPS
            ):

                sample_indices = rng.integers(
                    0,
                    n,
                    size=n
                )

                y_boot = (
                    y_array[
                        sample_indices
                    ]
                )

                if (
                    len(
                        np.unique(
                            y_boot
                        )
                    )
                    <
                    2
                ):
                    continue

                auc_a_boot = roc_auc_score(

                    y_boot,

                    prob_a[
                        sample_indices
                    ]
                )

                auc_b_boot = roc_auc_score(

                    y_boot,

                    prob_b[
                        sample_indices
                    ]
                )

                bootstrap_differences.append(

                    auc_a_boot
                    -
                    auc_b_boot
                )

            bootstrap_differences = np.array(
                bootstrap_differences
            )

            lower = np.percentile(
                bootstrap_differences,
                2.5
            )

            upper = np.percentile(
                bootstrap_differences,
                97.5
            )

            final_delta_rows.append({

                "Model":
                    model_name,

                "Comparison":
                    f"{config_a} vs {config_b}",

                "ROC_AUC_A":
                    auc_a,

                "ROC_AUC_B":
                    auc_b,

                "Delta_ROC_AUC":
                    observed_difference,

                "CI_95_Lower":
                    lower,

                "CI_95_Upper":
                    upper
            })

    final_test_auc_differences = pd.DataFrame(
        final_delta_rows
    )

    print("=" * 100)
    print("FINAL HELD-OUT AUC DIFFERENCES")
    print("=" * 100)

    display(
        final_test_auc_differences
        .round(5)
    )

    print(
        "\n✅ Paired held-out AUC differences calculated."
    )

except Exception as e:
    print("\n❌ STEP 78 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 19. Final Interpretation

The final interpretation integrates:

- baseline model performance;
- feature-group ablation;
- repeated statistical comparison;
- SHAP explainability;
- SHAP stability;
- calibration;
- held-out test performance.

The primary emphasis is placed on the source and robustness of predictive information rather than raw classification accuracy.

In [ ]:
# ============================================================
# STEP 79: AUTOMATIC FINAL RESULT SUMMARY
# ============================================================

try:
    print("=" * 105)
    print("FINAL RESEARCH FINDINGS")
    print("=" * 105)

    for model_name in FINAL_MODELS.keys():

        temp = final_test_results[
            final_test_results["Model"]
            ==
            model_name
        ]

        all_row = (
            temp[
                temp["Configuration"]
                ==
                "All Features"
            ]
            .iloc[0]
        )

        no_ocular_row = (
            temp[
                temp["Configuration"]
                ==
                "No Ocular Symptoms"
            ]
            .iloc[0]
        )

        ocular_row = (
            temp[
                temp["Configuration"]
                ==
                "Ocular Symptoms Only"
            ]
            .iloc[0]
        )

        drop_without_ocular = (
            all_row["ROC_AUC"]
            -
            no_ocular_row["ROC_AUC"]
        )

        difference_all_vs_ocular = (
            all_row["ROC_AUC"]
            -
            ocular_row["ROC_AUC"]
        )

        print(
            f"\n{model_name}"
        )

        print("-" * 70)

        print(
            f"All Features ROC-AUC       : "
            f"{all_row['ROC_AUC']:.4f}"
        )

        print(
            f"No Ocular ROC-AUC          : "
            f"{no_ocular_row['ROC_AUC']:.4f}"
        )

        print(
            f"Ocular Only ROC-AUC        : "
            f"{ocular_row['ROC_AUC']:.4f}"
        )

        print(
            f"AUC lost without ocular    : "
            f"{drop_without_ocular:+.4f}"
        )

        print(
            f"All minus ocular-only AUC  : "
            f"{difference_all_vs_ocular:+.4f}"
        )

        print(
            f"All Features Balanced Acc. : "
            f"{all_row['Balanced_Accuracy']:.4f}"
        )

        print(
            f"All Features Sensitivity   : "
            f"{all_row['Sensitivity_Recall']:.4f}"
        )

        print(
            f"All Features Specificity   : "
            f"{all_row['Specificity']:.4f}"
        )

    # --------------------------------------------------------
    # SHAP summary
    # --------------------------------------------------------
    ocular_group_row = (
        shap_group_importance[
            shap_group_importance[
                "Feature_Group"
            ]
            ==
            "Ocular_Symptoms"
        ]
        .iloc[0]
    )

    print("\n" + "=" * 105)
    print("EXPLAINABILITY")
    print("=" * 105)

    print(
        "\nOcular symptom share of total "
        "mean absolute SHAP attribution:"
    )

    print(
        f"{ocular_group_row['Relative_Contribution_Percent']:.2f}%"
    )

    print(
        "\nTop three SHAP features:"
    )

    for _, row in (
        shap_feature_importance
        .head(3)
        .iterrows()
    ):

        print(
            f"- {row['Feature']}"
        )

    # --------------------------------------------------------
    # Calibration summary
    # --------------------------------------------------------
    print("\n" + "=" * 105)
    print("CALIBRATION")
    print("=" * 105)

    display(
        calibration_summary[
            [
                "Model",
                "Brier_Score",
                "ECE_10_Bins",
                "Probability_Bias"
            ]
        ].round(4)
    )

    print(
        "\n✅ Final numerical interpretation completed."
    )

except Exception as e:
    print("\n❌ STEP 79 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 80: CREATE MASTER RESEARCH SUMMARY
# ============================================================

try:
    master_summary_rows = []

    # --------------------------------------------------------
    # Dataset information
    # --------------------------------------------------------
    master_summary_rows.extend([

        {
            "Section": "Dataset",
            "Finding": "Final observations",
            "Value": len(df_model)
        },

        {
            "Section": "Dataset",
            "Finding": "Number of predictors",
            "Value": 26
        },

        {
            "Section": "Dataset",
            "Finding": "Dry Eye prevalence (%)",
            "Value": y.mean() * 100
        }
    ])

    # --------------------------------------------------------
    # Final test model results
    # --------------------------------------------------------
    for _, row in (
        final_test_results.iterrows()
    ):

        master_summary_rows.append({

            "Section":
                "Final Test",

            "Finding":
                (
                    f"{row['Model']} | "
                    f"{row['Configuration']} | ROC-AUC"
                ),

            "Value":
                row["ROC_AUC"]
        })

    # --------------------------------------------------------
    # Main SHAP finding
    # --------------------------------------------------------
    master_summary_rows.append({

        "Section":
            "Explainability",

        "Finding":
            "Ocular symptom SHAP contribution (%)",

        "Value":
            ocular_group_row[
                "Relative_Contribution_Percent"
            ]
    })

    # --------------------------------------------------------
    # Stability
    # --------------------------------------------------------
    ocular_stable = (
        shap_stability_summary[
            shap_stability_summary[
                "Feature"
            ]
            .isin(
                FEATURE_GROUPS[
                    "Ocular_Symptoms"
                ]
            )
        ]
    )

    for _, row in ocular_stable.iterrows():

        master_summary_rows.append({

            "Section":
                "SHAP Stability",

            "Finding":
                (
                    f"{row['Feature']} "
                    f"Top-3 Frequency (%)"
                ),

            "Value":
                row[
                    "Top_3_Frequency_Percent"
                ]
        })

    # --------------------------------------------------------
    # Calibration
    # --------------------------------------------------------
    for _, row in calibration_summary.iterrows():

        master_summary_rows.append({

            "Section":
                "Calibration",

            "Finding":
                (
                    f"{row['Model']} ECE"
                ),

            "Value":
                row["ECE_10_Bins"]
        })

    master_research_summary = pd.DataFrame(
        master_summary_rows
    )

    display(
        master_research_summary
    )

    master_research_summary.to_csv(

        f"{OUT_DIR}/"
        "master_research_summary.csv",

        index=False
    )

    print(
        "\n✅ Master research summary saved."
    )

except Exception as e:
    print("\n❌ STEP 80 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 81: SAVE ALL FINAL TEST OUTPUTS
# ============================================================

try:
    final_output_files = {

        "final_test_results.csv":
            final_test_results,

        "final_test_predictions.csv":
            final_test_predictions,

        "final_paper_table.csv":
            paper_final_table,

        "final_auc_confidence_intervals.csv":
            final_auc_confidence_intervals,

        "final_test_auc_differences.csv":
            final_test_auc_differences
    }

    for filename, dataframe in (
        final_output_files.items()
    ):

        filepath = (
            f"{OUT_DIR}/"
            f"{filename}"
        )

        dataframe.to_csv(
            filepath,
            index=False
        )

        print(
            f"✅ Saved: {filename}"
        )

    print("\n" + "=" * 90)
    print("✅ ALL FINAL EXPERIMENTAL OUTPUTS SAVED")
    print("=" * 90)

except Exception as e:
    print("\n❌ STEP 81 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

In [ ]:
# ============================================================
# STEP 82: CREATE PAPER-READY FINAL INTERPRETATION
# ============================================================

try:
    interpretation_lines = []

    interpretation_lines.append(
        "FINAL RESEARCH INTERPRETATION"
    )

    interpretation_lines.append(
        "=" * 80
    )

    interpretation_lines.append(
        ""
    )

    interpretation_lines.append(
        f"The final analysis included {len(df_model):,} "
        f"observations and 26 predictors."
    )

    interpretation_lines.append(
        ""
    )

    for model_name in FINAL_MODELS.keys():

        temp = final_test_results[
            final_test_results["Model"]
            ==
            model_name
        ]

        all_row = (
            temp[
                temp["Configuration"]
                ==
                "All Features"
            ]
            .iloc[0]
        )

        no_row = (
            temp[
                temp["Configuration"]
                ==
                "No Ocular Symptoms"
            ]
            .iloc[0]
        )

        ocular_row = (
            temp[
                temp["Configuration"]
                ==
                "Ocular Symptoms Only"
            ]
            .iloc[0]
        )

        interpretation_lines.append(
            f"{model_name}:"
        )

        interpretation_lines.append(
            f"  Full-feature test ROC-AUC: "
            f"{all_row['ROC_AUC']:.4f}"
        )

        interpretation_lines.append(
            f"  No-ocular test ROC-AUC: "
            f"{no_row['ROC_AUC']:.4f}"
        )

        interpretation_lines.append(
            f"  Ocular-only test ROC-AUC: "
            f"{ocular_row['ROC_AUC']:.4f}"
        )

        interpretation_lines.append(
            f"  ROC-AUC reduction after removing ocular symptoms: "
            f"{all_row['ROC_AUC'] - no_row['ROC_AUC']:.4f}"
        )

        interpretation_lines.append(
            f"  Full minus ocular-only ROC-AUC difference: "
            f"{all_row['ROC_AUC'] - ocular_row['ROC_AUC']:.4f}"
        )

        interpretation_lines.append(
            ""
        )

    interpretation_lines.append(
        "Explainability:"
    )

    interpretation_lines.append(
        f"  Ocular symptoms accounted for "
        f"{ocular_group_row['Relative_Contribution_Percent']:.2f}% "
        f"of total mean absolute SHAP attribution."
    )

    interpretation_lines.append(
        "  Eye strain, redness, and itchiness/irritation "
        "were the three highest-ranked SHAP features."
    )

    interpretation_lines.append(
        "  These three features occupied the Top 3 positions "
        "across every repeated SHAP stability split."
    )

    interpretation_lines.append(
        ""
    )

    interpretation_lines.append(
        "Interpretive conclusion:"
    )

    interpretation_lines.append(
        "  The evaluated machine-learning models show only "
        "modest overall discrimination. However, feature-group "
        "ablation, repeated statistical comparison, SHAP analysis, "
        "and SHAP stability consistently indicate that most useful "
        "predictive information is associated with direct ocular "
        "symptoms rather than demographic, sleep, health, lifestyle, "
        "or digital-exposure predictors."
    )

    interpretation_lines.append(
        ""
    )

    interpretation_lines.append(
        "Important limitation:"
    )

    interpretation_lines.append(
        "  These results demonstrate predictive associations within "
        "the evaluated dataset. They do not establish causality, "
        "clinical diagnostic validity, or suitability for clinical deployment."
    )

    interpretation_text = "\n".join(
        interpretation_lines
    )

    print(
        interpretation_text
    )

    interpretation_path = (
        f"{OUT_DIR}/"
        "final_research_interpretation.txt"
    )

    with open(
        interpretation_path,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(
            interpretation_text
        )

    print(
        f"\n✅ Saved: {interpretation_path}"
    )

except Exception as e:
    print("\n❌ STEP 82 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))# ============================================================
# STEP 82: CREATE PAPER-READY FINAL INTERPRETATION
# ============================================================

try:
    interpretation_lines = []

    interpretation_lines.append(
        "FINAL RESEARCH INTERPRETATION"
    )

    interpretation_lines.append(
        "=" * 80
    )

    interpretation_lines.append(
        ""
    )

    interpretation_lines.append(
        f"The final analysis included {len(df_model):,} "
        f"observations and 26 predictors."
    )

    interpretation_lines.append(
        ""
    )

    for model_name in FINAL_MODELS.keys():

        temp = final_test_results[
            final_test_results["Model"]
            ==
            model_name
        ]

        all_row = (
            temp[
                temp["Configuration"]
                ==
                "All Features"
            ]
            .iloc[0]
        )

        no_row = (
            temp[
                temp["Configuration"]
                ==
                "No Ocular Symptoms"
            ]
            .iloc[0]
        )

        ocular_row = (
            temp[
                temp["Configuration"]
                ==
                "Ocular Symptoms Only"
            ]
            .iloc[0]
        )

        interpretation_lines.append(
            f"{model_name}:"
        )

        interpretation_lines.append(
            f"  Full-feature test ROC-AUC: "
            f"{all_row['ROC_AUC']:.4f}"
        )

        interpretation_lines.append(
            f"  No-ocular test ROC-AUC: "
            f"{no_row['ROC_AUC']:.4f}"
        )

        interpretation_lines.append(
            f"  Ocular-only test ROC-AUC: "
            f"{ocular_row['ROC_AUC']:.4f}"
        )

        interpretation_lines.append(
            f"  ROC-AUC reduction after removing ocular symptoms: "
            f"{all_row['ROC_AUC'] - no_row['ROC_AUC']:.4f}"
        )

        interpretation_lines.append(
            f"  Full minus ocular-only ROC-AUC difference: "
            f"{all_row['ROC_AUC'] - ocular_row['ROC_AUC']:.4f}"
        )

        interpretation_lines.append(
            ""
        )

    interpretation_lines.append(
        "Explainability:"
    )

    interpretation_lines.append(
        f"  Ocular symptoms accounted for "
        f"{ocular_group_row['Relative_Contribution_Percent']:.2f}% "
        f"of total mean absolute SHAP attribution."
    )

    interpretation_lines.append(
        "  Eye strain, redness, and itchiness/irritation "
        "were the three highest-ranked SHAP features."
    )

    interpretation_lines.append(
        "  These three features occupied the Top 3 positions "
        "across every repeated SHAP stability split."
    )

    interpretation_lines.append(
        ""
    )

    interpretation_lines.append(
        "Interpretive conclusion:"
    )

    interpretation_lines.append(
        "  The evaluated machine-learning models show only "
        "modest overall discrimination. However, feature-group "
        "ablation, repeated statistical comparison, SHAP analysis, "
        "and SHAP stability consistently indicate that most useful "
        "predictive information is associated with direct ocular "
        "symptoms rather than demographic, sleep, health, lifestyle, "
        "or digital-exposure predictors."
    )

    interpretation_lines.append(
        ""
    )

    interpretation_lines.append(
        "Important limitation:"
    )

    interpretation_lines.append(
        "  These results demonstrate predictive associations within "
        "the evaluated dataset. They do not establish causality, "
        "clinical diagnostic validity, or suitability for clinical deployment."
    )

    interpretation_text = "\n".join(
        interpretation_lines
    )

    print(
        interpretation_text
    )

    interpretation_path = (
        f"{OUT_DIR}/"
        "final_research_interpretation.txt"
    )

    with open(
        interpretation_path,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(
            interpretation_text
        )

    print(
        f"\n✅ Saved: {interpretation_path}"
    )

except Exception as e:
    print("\n❌ STEP 82 FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))

## 20. Experimental Workflow Completion

The complete experimental workflow now includes:

- Data-quality auditing
- Blood-pressure validation and preprocessing
- Exploratory data analysis
- Semantic feature grouping
- Baseline machine-learning models
- Feature-group ablation
- Incremental feature-group analysis
- Repeated statistical comparison
- SHAP explainability
- SHAP feature-importance stability
- Calibration analysis
- Final held-out test evaluation
- Bootstrap uncertainty analysis
- Final interpretation

The analysis is complete and no further model optimization is performed after observing the held-out test results.

In [ ]:
# ============================================================
# STEP 83: FINAL PROJECT COMPLETION CHECK
# ============================================================

try:
    required_final_objects = [

        "final_test_results",
        "final_test_predictions",
        "final_auc_confidence_intervals",
        "final_test_auc_differences",
        "master_research_summary"
    ]

    missing_objects = [

        name
        for name in required_final_objects

        if name not in globals()
    ]

    if missing_objects:

        raise ValueError(
            f"Missing final outputs: "
            f"{missing_objects}"
        )

    print("=" * 95)
    print("DRY EYE MACHINE-LEARNING EXPERIMENT — COMPLETE")
    print("=" * 95)

    print("\n✅ Data cleaning")
    print("✅ EDA")
    print("✅ Feature grouping")
    print("✅ Baseline modeling")
    print("✅ Feature ablation")
    print("✅ Incremental analysis")
    print("✅ Statistical comparison")
    print("✅ SHAP explainability")
    print("✅ SHAP stability")
    print("✅ Calibration")
    print("✅ Final held-out evaluation")
    print("✅ Bootstrap uncertainty")
    print("✅ Final interpretation")
    print("✅ Results exported")

    print(
        "\n🎓 EXPERIMENTAL NOTEBOOK IS COMPLETE."
    )

except Exception as e:
    print("\n❌ FINAL COMPLETION CHECK FAILED.")
    print("Error type :", type(e).__name__)
    print("Error      :", str(e))